# AP Model 3D Foto MAX: usa toda la GPU y la RAM (A100 / L4 / T4)

Foto → modelo 3D con textura HD, **esqueleto automático compatible con Mixamo**,
**pesos sin huecos**, **6 animaciones**, piezas por hueso y exportación a
**Blender (.blend) / FBX / GLB animado / OBJ / STL / video MP4**.

**Qué hace distinto esta versión**
* Se ajusta sola al equipo. Con una **A100 de 80 GB + 167 GB de RAM** usa el perfil **MÁXIMO**: forma octree 512, hasta 150 000 caras (antes ~25 000), textura de 4096 px y video de 1080p.
* **Usa la RAM a fondo**: carga en la RAM los pesos de los modelos (cerca de 30 GB) y deja **todos los modelos cargados a la vez** en la GPU (1 foto, varias vistas y textura). Trabaja en `/dev/shm`, un disco en RAM, guarda en caché los resultados y deja que Blender corra **en paralelo** con la siguiente generación.
* **Pesos sin manchas**: los calcula Blender (método de calor) sobre una copia cerrada de la malla y los pasa al modelo real. Cada hueso mueve su zona completa, no solo una banda.
* **Turbo**: forma con 5 pasos + FlashVDM, textura turbo y Blender en dos procesos a la vez (archivos y videos). Barra con **porcentaje y tiempo restante** y **monitor en vivo** de GPU/VRAM/RAM/CPU/temperatura que se pone **en rojo** si algo se satura o hay cola.
* **Ves el modelo antes**: el estudio se abre en cuanto está el GLB animado, mientras los videos siguen renderizándose. El visor usa un **GLB ligero** (Draco + JPEG) que carga mucho más rápido en el celular. La GPU se **precalienta** al cargar y los tiempos aprendidos se guardan entre sesiones.
* **Esqueleto detallado (66 huesos)**: dedos de las manos (5 × 3 falanges, nombres de Mixamo), dedos de los pies, ojos y mandíbula, colocados midiendo la malla. Animaciones nuevas **Manos** y **Hablar**; los dedos se relajan o cierran en las demás.
* **Por etapas, de más a menos**: primero la malla completa de la IA en alta resolución (`malla_alta.glb`, pestaña *Alta*), luego se reduce a las caras pedidas para la textura y el esqueleto.
* **Física de telas**: túnicas, capas, mangas y faldas se liberan de piernas y brazos y cuelgan de la cadera o la espalda, para que no se rasguen al animar. Los vértices duplicados se sueldan y los pesos se calculan con hasta 3 intentos.
* **Huesos de tela con movimiento secundario**: en túnicas, faldas y capas se crean 4 cadenas de huesos (frente, atrás, izquierda, derecha) que se balancean con retraso e inercia en cada animación. Queda grabado en GLB, FBX y .blend.
* **Piezas con nombre**: Cabeza, Torso, Brazo izq., Mano der., Dedos pie izq.…
* **Aprende de ti**: bajo cada modelo, **¿Salió bien?** con 👍/👎 para forma, textura y esqueleto, más un comentario. Si corriges el tipo (persona/animal/cabeza/objeto), rehace el esqueleto al momento y **recuerda la corrección**: los modelos parecidos salen bien solos. La memoria se guarda en Drive si está montado (celda 6).
* **🎲 4 variantes de forma** en segundos para elegir antes de texturizar (contra la aleatoriedad que critican los desarrolladores).
* **PBR básico** (rugosidad estimada, metal = 0), **PLY con color** para impresión multicolor y **malla en cuadrados también en FBX**.
* **Panel para celular**: visor arriba, botón Generar siempre a mano y estado de la GPU en la cabecera.
* **🔁 Reintentar con lo aprendido**: según tus 👎 cambia la semilla, pasa al motor que mejor te sale, quita «Textura rápida» o usa el esqueleto simple, y vuelve a generar solo.
* **Revisión de UV** en el resumen: textura aprovechada, triángulos estirados e invertidos.
* **Malla en cuadrados limpia** (como *Smart Mesh* de Tripo): retopología QuadriFlow (o rejilla de voxeles si falla), UV nuevas y **color + relieve horneados** desde la malla original para no perder detalle. Con esqueleto y animaciones (`modelo_quads.glb`).
* **Detecta qué es**: persona, animal de 4 patas, **cabeza/busto** (mandíbula, ojos y cuello: Hablar, Mirar, Asentir, Negar, Sorpresa) u **objeto** (Flotar, Girar, Rebotar, Bailar, Saludar).
* **Texto a 3D** (como Tripo): escribe una descripción y se crea la imagen con HunyuanDiT y luego el modelo.
* **Textura rápida** (opción en Calidad, activa con Turbo): hornea a 1024 en lugar de 2048.
* **Dos motores de forma**: Hunyuan3D-2 (Turbo, segundos) o **TripoSG** del equipo de Tripo AI (más detalle). Blender **4.5 LTS**.
* **Malla reparada**: quita trozos sueltos, cierra agujeros y orienta las caras.
* **Cero vértices sin peso**: si alguno queda suelto, copia el peso de su vecino; el informe dice cuántos vértices mueve cada hueso.
* **LOD para juegos**: además del modelo completo, versiones al 50 % y 15 % con el mismo esqueleto y animaciones.
* **Visor web** (`visor.html` en el ZIP): botones para cada animación, cambio de calidad y modo AR en el celular.
* **Multivista**: si subes espalda o perfiles (o una hoja de vistas), usa Hunyuan3D-2mv y pinta también la espalda.

## Cómo usarlo
1. *Entorno de ejecución › Cambiar tipo de entorno de ejecución* → **GPU (A100 recomendado)** y **RAM alta**.
2. Ejecuta las celdas **1, 2 y 3** en orden (la primera vez tardan unos minutos).
3. **Opcional pero recomendado:** ejecuta la **3b**. Genera un modelo de ejemplo sin página web y muestra los tiempos. Si esa celda termina con "LISTO", el motor funciona.
4. Abre la **5. Página web** (o la 4 dentro del notebook). Verás tres pestañas que se llenan en orden: **1 · Malla → 2 · Textura → 3 · Estudio animado**.
5. Para no perder tus modelos, usa la **6. Guardar en Drive**.

**Para actualizar a un notebook nuevo sin reiniciar:** ejecuta la celda **1** (cambia el código y conserva los modelos cargados) y después la **5**.

## 1. Proyecto (código incluido: no necesita Drive ni subir nada)

In [ ]:
# @title 1. Proyecto (código incluido: no necesita Drive ni subir nada)
import base64, json, os, subprocess, sys, zlib
APP = "/content/ap3d_app"
os.makedirs(f"{APP}/ap3d", exist_ok=True)
_FILES = json.loads(r"""{"__init__.py": "eNoDAAAAAAE=", "config.py": "eNq9Vttu4zYQfddXEMpD5F1Bli9xXBcq4FyaBkiaIE2AAkEgMBJtayOJXJLyJi32u/reL+sMScXybpLdfakAWxQ5PDNnZjgc3/ePmGaZpqSk5OTyhjzh4Gp+DgNWFksG/0QwuShKUlFFaKk5+dgwktF7FnneIa9JU1MyH8QxyRmZxuTkAPYOJvs4gBnEUow0inagzud/np5fzDzNc65ICb+K5wzfIArwOa1Bg1xSXGZ1a5yxcs3+Cs0WwRRXHmwQkhnh2omiyiCj2YqhAYIuixpMf0/6OVv31arqWZYVLUsKUNTT7FE3khrOZF3kjBNFSwAzCiXoKZusAKrOBTTyfN/3ikpwqQkY4UZq1eii9NpPoezn1cXFNUlALmL1upC8jpZMB/78cnSU4pofEr+f8VqzWvepGOV+z7u4uU6PTq9e2wbLX+1KweYipwp2H5wd/350/Oput7yFcA98cybbN4B4Xs4WJF2KJujNPAKPlk92gI8jqbnMVpvJhZ2IsianUaFSuqZFSQG0xWgfAcZ1RMG4FMJTZCwVkkOa6IKpIO5t7ZEMwlQTEdW0YiG8Nde0TCtWcflE+mT47t0oNjvYY8aEJsfmBZHb6BZUKa8D5h9e3oAj4ih2hCWt0rxQD0FV1OlCMpYu75Opsx7CfkilYHBgILW0pPf0A8e0Mzm3STBVkBWFdFKCQuL8jN81DwngZjzC3PnKnagJXGJTKEIDUjgzSxb4Larfi4xQl6fzuZn/JSEdk7e9nQP0M5DNFme8vyUH6VLRB5YXUgV5CG4slE75Q3ItG/ZiLEAIFX6v03ObkoLqVfSBF3WA+Q9p2BpjdXzbio32TeByU8naPLMpsoZogkqbxHanmbBHM4KDoZvnDAp6NqG2PJyJRlmjYZRmvKlBBeGSDM3yJy4fQohalboAPqcPHCAXHWMERGc/nll37Nh6aYtln/wGH173XOQF8LClMvFtrYQU5ZkGFcneYAgKNRMq2YtDkq2a+kEl4xifcCtENq9oxlQy2DPLBCodiP40CcGDeMyTYTyehlDTypTX6CXj4xdg+GJRcponv9JSoV+xSKYSkeMp4CpaiRK+huPQuIRJlYysw+Ea2bhgNJ11HDBGB7zOfH523eE9RkO/4j38Bu/h/897+ALvYex4n43JcPwm6/Pjo9MO7dF03NIeb2hP3mQ93ZC2PF8jbVn9AOv9YYf0YPICacVmX4DtkOsxGey9Sfpg/sfpYYf1cG/Ssh5tWE/fIj1+hfQgxuD8AGmTCd/DeeCO+A7BNqhqshVt251lQ6VpYrgooGeAu6Lb5MB9AbcOVEXoOWruMNa8XDOJLQdjEqVA1l4Y0Jdgl8NkVcBVbZoQVy8N8oG9sR0MCCBuCYpI4JZM94Ua36N9IegEwyoBaPBuu6teZBDErY8lzLRPqTFY+Xe2rmEiT+JWCroudFf6iRXLld4SGsdt6XMTcAw7l8Gt7zxo9lT0MehOhcQVDhE1IqeaBRi0TTnH61hCGc4D/ArJoBeSrel2djvCWMcT/LPxgxotk+3ivUFoZxBm674Rz1eNymRxzwLhbhu3Hix87FP/Fre7YPTu3WcS4NiZjd8nBz3y7z8mTXClu4Dz27fxwkcZNBolMIAg4lpoXLFDXIN5e3LMvB26+S8hV1RB94Ji5tzs3s3Cz9hsQ1aBdNsK4zqMEUM8vgTT6X1QtvUpbIBL/D/sVZwi", "engine.py": "eNrVfWtz20aW6Hf9il64cgM4FExKfkUbptaO5MR7ZVvXdpLZ0ahQINEkYYEAggctyuX9W/t9f9k9j26g8aIk2zt1b2rGIoB+93mf06cty3qVFEl2JBbwx84dsf+zWPtR5IvDY2H/Vsbb0o8Pj/cPxANxsN7Q90JeFWXm48/nkYwDmbl7e7/niQikyOS8zPIkP9oT4r54nwRJLiL4/zoJJP79q5SBH4u5ny19/CZjAX39eva78PHHRl4LO5XZIozEq2d/e/nqjXgALQnx7PT9G2ck/DwUcSJW/lbIHMr52EKRSVHGPk1BbLGZPFyWIXyQLg3jFDpKJYxK5FKkMETsPVZ9v332Sthzf76SOP7UX4YxtAo/T8O4vHJEJLcwx4RGgTPws/kq3PDQUz/zI5jXkVCNZjAQNVdR+FkAQ5HLEqrnPJKTSBSZP/M/wGe5ThOoLjY+NvUgkJsH+Wot7CDM5wm+gpE5agK8RDT6pJBLKeZJjF3N/TiAdfxXvRHC/gXK/UBVoUwGKyNjGroeKtU0l0gsZQwLOQ+TuN6Net0ymZdRQXsF3S9LnBQVxBXnZeOhCpgNbEy+wrWL9G7Uy/ahzAuZ40qlsggzLLIO87UqKHMRxnnhx/A/mbh7lmXthbhABQ4YYCqDsbqLEgAPisIGzRf6+3Je/YqSmf6NI4nC6jFM9K8PeRLr30muf2VS/8pXZRFG1dO2KpKXM1j9ucyrN8Uqk34QxsvqRbiu2rkOU4Biuaef43KdbnHkcbq3yJK1OHt5KtS3l2t/KUf8502a73EBV9QrsAiXe3u/nbw9EVMYtZv6xcoNwiz219LWz/4sx7+252HHnuc4e7CV3umbX/431KoG654m80vb2fvj5cmf7+CDbUFvcWGNhDXz55f4N5ILes7C5aqwnL29vXviHQByFgK0HcGwfASaXCxDwCPYVZGIhR/lOLlEpIDiABJFuZ4BPsgI4CbbhEGS7dH8XMBr7+WrZ7+eeGcv/3ZyikN4/NAbj8f4/z38+u7lMc7zYPzwKXQdyIXI/YX0qFs7XC+dIwIrgJE//AiGBEgfJxmQrfDaJ7jkAeKO4Ve78Nd+nIwQMeHfBRYtEsdFGMN2wgVUBxDJGQLn1IfaDZf+VR3if5kfAh5Av6U8ybIksy3AaUUTcPawGMYINjQ+WEGs+nEkVjAtaNzNw2up+/4ofhKHBwKxh37t6uu0alkijVoDPQTUBOIGdBV6tddhHK4TbC69clS31MV9aPxn0b8BX9DhMgPCI+sO1v6VjdNzoBO9g3WzMGM173mSbm3H/OAWKwCU2A8j29Y1R1Ubjt6G02evf/n7m3dVh1gT6SzvXAxA/PbX5wiy8OeZ5fT3HW9kVtiqCJXIJBCUGD8rQJtH0o/L1EvKIi2L3L6UMp0+HQPshLG3yKT0lrPpZFxD4HMgsj5ROINSwjKJTSg/INEMiVtVS6dwpfAF0fkKCItsWw86yGHMOaC+DOwAAVYEOEmkby7+U6H8hySMbSYP7pvf33vHL9/CGty3HAcXSZcKcyAVduDUC48zwS6I1gEhyS+9MkfsarbluFQQWP/9+4fjqvbHFdAXHCQAgbCB89hBjjuPi4VwTJV+MpfM2BCaH/QdwOCS1B47jS9qRNkamLq0g5EIl4Da0pMIjvn0fVbKZgVjmvIqzGHLAuB/lgvk12r1iv9B2Uyuk400izVKffnSyKu5TAtxQn+Anda9pz4wDQawv4DrFl6ULHO7BqI3c4Qc4W/CnGUkpCRA8uZAjgSICBVoAS1LgNa4gH7IMKDdBNGS2gnCRZlDt4Ayr1+8QS682h4GgLoOMX9qVmJbS1imLBHrUuYoOuUJSAXwwd8g7aupIvMeGOkSOZzx6qOfxfAqZ6KmHlzgOYXM9KNt8cYBSq6BY8L6TS33vjF4976mijc2MPdB5Emy7fQFsf8/uQTXRtxAJsg0QE0Y6YD66eYrP+15JVvvQKbFNwbEqIm7S1mcwk+Z2diP4+bwLDcysnWBP5+9ff3y9a9OF42Ji8O2wL7ILHcRoPLWuqJAEMDvGvXhAfvwgFbNkjwstgz7imreAGP9QwCRM86R7e0aRWGOoviqUTCke2tAtCz0PRJK1Mp2B+eqYnpIWLpmSEyfuYUbulZlAXy5/4wkYg/kfSDFH20Aw9VIIOOdHj59CKrE9fRw7BqU/CXzuYCVCVaCQCQgyRv43SKc+0cof2/lnIRlGGyi3qOERs1c78/KxQLkcGgFFAJYVtBHymuSjVlEATZqEzrKCBAxAwkB5o4yVC2PbID8xKnr5z6wli0N3EXOFYLsORJBsU3lFL4vosQvHj9UeNBTZ+FDBf7s8+cMBEAABdu/5tdvVS2qc34OP+cJfAWuCxwPnmD68HQxEufwPIG3+HO/+qCLcaWLC6ca/kb8m3jrvlfP+/ACttmPNbVH9jZ2fwSpBPcDiCjKD9BQWqT25vwIO3Ko5erFBF9M5P5jjWYh9nK+uKDHmOcxz5I8t+Eb1xD7Qv3GcaufB43Xjqr+gOpHoPpFSxc3ygYy6gM/mcKska0F4VpxH2AbMI7JAWs2sLfcdRSmdgyzrpdz7B4+gj5c+ufphQPTnLhjWrOJA1Mfu08eQWNjl5uao4yehbxOEx5YeoXy+ZzHiqsF5dWKHTiw2QgKdhgXqvQWS+vvMM25Wgeo2C2dXGJp6OFn2AtH/C/6/RM1z09b48tWf+GqGaqaCnSWKKfwGA9gktV/9xAH/AJxYAXaJXA30IoR0vdqsQyhuIxA8sO2GTkRqNzxj9zR9Wyw0P5EqkI4EoQf2CmYXDX/g7FTs4gQ+QMN+zy5PKcfFxc16biCPdmOoZ306jwEWEm38McUIc+xAy60LzLnCP7+AB3/gEihPl2pT1fVpwtoEABEN5Ut1WxAyQAF7ByXAKHgKQBI9fsp/b7A+TUkVJaCkWAydNka5qDVGqQOHj2qNhoKlLAeTx1U3+YR0GVxEgOxlzxtotAeqAuF59m5jBYjpG2L6eskhkUG8j9NM4SVepGwkJviIkE5FPOUQBTIAgQVQ6qngtACTt/gKfQahK8ZfsBuml+IKw9/8dabHR+LnRUHPqc+THD4i3cZxiioKqpcffyQzJB+fbL8eYGGIOsI19+SsTdPIh+fPjcraBYElV6Allz3dU8UIVqBQGGQQYi6gY0/gPkkmzBSYhsIaaCBwWqXgDxKd2i2LwufxkOKLfT/FKmMpax08Hz4iF7M2EQELx4DhrXHCI0QozbsC/3KhRqyi3YUQ3RuMHZzaG6ZBiDA2VgeoMIP7CQFjaHZq2PoJsMcviHmVH2wEQoW4HMb1Pxg21lz+qSWwkuTJIIS84X7nqwjZ/B8ciXnJQiqNuC19zHJLkFomh4iDSXIOLfUO+uiHjIsx9q/hK3L8s6CkVbiJZdKd6mRT8Y5SLO0EAaWgRDwLkRxfJ4E4TJBoxvAWYm2DWW7M6ypW7+ypIKYnKzTCGEBvlTNAQEOZzCdXMSlRKslwVSM5scYaDjqBJkEKgAEmkxyuUQVohJFtLLGu+UF4RzIRf0FRUSYiw/KiW0pTASpGhHK2VkKcfKGgjUOopxuDRVDbIQCu7BxqKrGS6hOMDJUDtaUergbfg32qmEe2rwLpg21R+CP4xvsEUAgTW5Y7eIg7JQA1TpA9ce2THyxHBHmVLKl0583i118DWLtMXnc/4r/GDMqdIO9RuLjfZRo08wVy8skLPAUpi9B9Y2LB4bDw6I97rxeb2ArGuh6KqUoKkdH5R5AZwL5HAjjAvnBz6LK4fH22asGjpEmtCpJGUOx3VuVM60Q5XMfgBl3GfAvq6uEqEtMxfnFMA1G0SdD0afZhu24NPGueQS2PKNvXkgWp4FiVdtyQ6VcRCQkHANl6/H+AAPOi8xeuGSjJtpPTS24uw29zy8GuQGqqrKH04CkYVuviV6mZZCISEoiaJVn57cXR7ClLeuRkq7GezVaoOYY2DSwZjeoZIwbbz6GwC+Jm2FxtJfPoAvWAGEjpyA8w2gX3TVhAxoyhP71QkGRGZj9+KH46Sdx0DKTGfuFBtDZ8LKrGca9BWLcD7TfzZx6BVAkQkeGi/8YYh3Nth+jJ49ppvKqOY4iKXzkr3m5tuUVsEgQV4MRQ4LTlRfthXXGGKO8acpBCMhyJD5xY8reduROFp/Fr8/xs9Wd2sL6ZMwAFb4jd7z4nBtcRK0LtWowZWZPaZhKg0KM0JMAGmSZzRKnZb5oG5U01lYk4zh8/yJKPr7yCyAM8fIMmgY9sxZIsC9Yo5vKk+yPHAs0KngM7BYcq1Eukggo8JTGW+bSQ8cJULG8azwF0KEJNbfsnngBqsLqj2NY8kCiDEL2jUgbRP5zMr4i83bmp6Hyig6SHz09V8Y+8AZvgU1vgrW9nnugaCdTaz1vWV8Vur8HDYY8EN0Ge3sZ7KmLNrcgKF3CIqplQdkpCPM0iUPoqIek3LL9O7etTWEwyxpeSZ5mQP0I6gHIVbTL6olEKPMFbbj54i8ULIstaX0GZJtGapPV/YJYaUqgrniPTQr7kbJF/1BNxkGPEcix6HQj+xeMF/REA97ISQwilcj9iLzIla9c/JJUdjeQKEThr2chFKDoAS3yQuNVU8rlziY2GM2Yh6NVKDWKX8gbFzQ4r/bcomW6JoFkp0ZvKAg0uTsvAx+oV7EuIxewIPnoFYvDA0BaXMkdleL4rsVnIHCsQNu7bJdu0BpWotXIn0MLyyyBhXtLjo3MxHBzk/vFtkaJaSWLwbi9JPaWaQmCHBkvCwOEbmYW0Leh7/f23LAHdGbRbKvumnxODc1/uHFCL4JZgCphBrMoqGXwI1LmO67rtohRL6lpWR1YMzLYRr8sOcifh/6zVqp6sB+Exf7mYH+8TwsAVKHrBfsCosNLcBM16wGRBlwCfwBZBxl1QxJtg15z0/q3TIOMsa8Idd2CjX5dPwUBLLCVduZoNsbT26IJP9vIIyAahPtIZg6AHkV+3GReEhTAXQvWC0YMQCPdeA8Idcxbt4SXvv3v6ql6mdcbWuFhLCEb2BcgCjSsUIX8FOjVzgs//wJcUVa4W00fdKy7I0wvzqw33xZp6gXZhTitbbEbqADEdGBjnO6u3QZTdBN3QRbUYr8a/GF4TbC4CREGRbe2mfd/FEr64aSLXF8pMOLyNIBkhLppmaNAQi73CUW+DYAO26YreODHXmBoSAbsxu7oIGdYvaN50IhLMmbb58aaUGcmx7losWAEFqD26Arrr2dddMwQORo0KCQr3wFniz5Ao9EL+9P31Pv3OJjqJ7XIA/qepb/vP98V+gY/dDwF/Qu6WzXrAF8/vW/obv3K/s26kQFFFm+dWqGjnUOisK9vugK9k3T2djevHS4w3n6rSCb9y2+FnrcRfHZ7ikxKXOHnMBFWNA30KrYcg24eXrOVAQVueGl6FDodnFvJAj3+gXVxdMNAtRJMSpE3T0tPVbXvRvNBRlSR3i+fNddL2HOgYJmKhqDApDro17Ea5AxXpE+r6FoqUwriafCCUZvRmS+cXvtlam4DUdDqjSleDtgQ3CKxLVgyy+l3JjWEX/ry0c/WZdpUVyqG2zKRo4UpQmeYcwe7BiIzC7e4PgNN/3PkzfawgG3eeUz/FA5/dy7fBP1XysFGYTysOBhGhQikHTZNKosF6xPWN5Md1t3uAf+0TbSHWtXB3WydtQsoK4spQdtIBD4o1TEbADH8ICtapKDunKVbmqFLBwCMsegVQEGmPkPRWAPX6rXr6vWsV+5GA62ujWFktuE3VfjWdZv+Hvs4PD/AgwVxCIq9H/EwsyNyjoYR264uZRYD7cMZ/fL78TOxBUoYLqUR+ZcLNE9mYbFWm1Aev37Np0+gjTSDUWf6hIOPBzh89ADh8ZUomd/OxKTMvg2Nt6EamPSECivpT1tj2H5Epqkw9/yND9MD0twmLWw2vNFe03Ud9R1ROM78j03+t4b2ODYllh9VaPUIcGVyMBLwjwO/OeQJ/+c4nSjgqlkX/7HDdauIK6MoTHNp25Mnj0fiITRzeAg/DsZjhzwJ0dSG3yMxeYL/PIJ/MAim3UpGobTLCNv5EapPfsSGJtjik7qhx/ASY70nj3ubQTsVr/qvdGQGPR+B3IRzObVwHyzHXfsxqHnA5UEGarlsyH3C1cMY6CWSO2LRNvTP7/2ySOZ+jk5dbE7H+vFHCvebPHb6Tew2RelPw/UID5p4dQ95IdN8CruRzDGk2sskKB8l0qLp5ODpAHnFJuZAPi/zqWbd/GhdjNR5IRjUdAmtUpy8R+NERzMGHrYtAzWg5tt4vspAhriWQwRoYaHdF6ge8AJE5duQijv5CIVufI3HnxKRrMMCeJciqhWZYZxU0ZnaXUwHZWCFpw9RDpHBWIVLcUi5N1sq2/oaz9RNNQOymnTqD1aZiaUrPwr8tYnw6m0C6HvsULhHLtcAmxglDKQThDufHcr13DmMFMjSjI6A8HkctJmT7BZe+5kr3kqU3DAej1yiQNjQqQPPGP8RoXxXG/R88yCD6pRJ9hz3A55w2Y6E/1cZEqncQH/s/1aENO+jfmjIB4oCw/eydugRB8L0S40kePXuZUOexTgOjsriw0xHSubMZGqrXas2ydEnnY4Ebx+fd6qe+NQTP5oxRZKOZZxTkCFuPuq/obgvnvw4+ZEFdXjZkD5JL/UyjHgPsBqSlfv3DycAxZMeEaSKVKQIeTt2ajUa8KxpYSVyoo93dWXpnNz3NOYuweBQaK8phtm0hEC1kfQBYcwD+I2BLQZI30Gp3ukcfeHP5VsZlHOZgUCGhE1mbe9Fe8AYydUoaTsU5jwonaJf24iDFj8D94D/hmVU3Us9ONUFxXh62AwQxim1crOoN6BgmOFrxuZqy5yNy94TuW4GyClPINSqyRViu1ck6oxcFc+5TgumVNNxkwihZpfgCd6winrPJPOM0HfFi9Pf/+ZO9vP5CiSlSACLPQCS95DFvxGIcXMqWLX4LKUwiwOEF/TZVYbAfxV8YleC1DnS1oPj8L0r3ikZEqU0tIpFyS2EJpoRwm2R2fyA8mwWgv51fgQbU+MHhgxTgDNX+UGgjSVEEUDMV0BA5wBIKEolsw8gG4y4/CwJtiOBagisRKDoPdGWUU+YgQUSLyDZx1VYSDGrHFjQT7IA6lWUQYiSNBCTEZ6m5l2F91u0Ba7gPax6AWJbU18Gsd0vYHq8h2ZgVr/9rxcD+VSUjDchzIBjuJ6dHR577w9eWo6YTivlyOoHUz4I+LaMkeuqo4AIE3Qgwp8DPqH3jI6xFcklw89vHL5E+OP06ELN0zAVKYjKq17L5ID9uQ2ZBQGyPqU4YPpTwrbZV9dkNYtgC/eBeAKs7oMsnT9o9vWFuihBsGdKcjMtyu2y9mwyf+0tZ9YFRuE/Hu8IeSCLRVmdMr3ZwtMOlrjJTlTtBMAinxeGdQSgxDa+PvYBoarXTm5QiwH7XNv0DeNDKGiZv6H+bgDT09IoMeqp2W/TBTJKRot9VeFBXXF/MwHQOdbgDr9AkooiGZiRnGjwHDVUQRiLSQzYIgroSkt+dKM6wqarhhqixRXn1upId8OM2BMbCSUQ+RLIO+kXKE1Px0j4+xQP4BoriricTpCF7MSgj2FQrLgcMdycTjVDU8DJl/CFxBJD+3Bc2u783CD75khN0t9L3pl+15TbUrzSWLJGpI1HkbT5UoeGJUkxtR6AlI4i8YP3+PHdry1pX70V9h/P3r0XDwQ9i2cvnSMV/IJSOdLRRQhYSKaeQObwkt1DN/PEHpZhxvsOG6XVUjWDUtrMBANCYZr6mHW+5YjlHpze6sPGcY6nrMe8Ps5tCjbCoLEWTCIHLSUtcstxumYJtQ1uqpAzr37pHdJLpVa/F/+pqRnoYpkZTQPPb189/7WJK+1DEWqMNffQYcVWj1ZLQ/IyFeei2u+wn0b7KG5gqf2J+xBWwCDxrtz4kT3UCxmoGzO+uZ8KbKmbrpCjTBFNI8SQ8VABO5n5hE06AwJ4BekV9FvOjbB4W61ez7zjnOmOayefOTweDhDmMyQt40CTGlDOCCJ6TEVqGtnIRsAYLNecHGQ3ZquXbFVphoIR5fNUPpLq6Cxou36mEmU0NUXdo/vaX8vgPeeeybYv4JWdl4tFeDW13DReotFJRrKQU7aTUwBzC90pgYW/AVawculMdOuADh0iUt+GKUsaRnSWzBiyCqOeLT1yL03rs5R0cFL/g0dBEZ28WBbTFooZKQ7wMGfU6rQ+/k8B3ntfaNrr4am35qdJxepx0MpuB4vRMqz1MVPsFk8bKig7eOQ4Ozlqi0kT8tr6JTEO/ftn8YiNFU/cMRCC3MejRH2MVQGj+57/2gmUMU4cUheHB3x418+RNYbLMinVseTkfHJhMtV+dyif8dRqcDs2teNyHeJsxtBjPq+K66cetKkdJG1+Yzpf+43hzRgLFzUZ3J0iJCGaW5GZi8uinjRevoJXb+nN3oDHVp39MZcCwURniRmuoUwFqkK2s5NqHerx2OoAkGkW5mIjYQ5mWh/+bdruoJA+dKw3zPnqsxtzDPfl3EGcuqe2Z/ptO632ELX8QV1BRpHInLI79Xxp0l6yKaJzgyu4oEjjITzOabC1hxwDHClMZTwM/cXUFaBO6WMLFHQ/5LdhBBwPmcUxxVZJ3q1Pm7ULSkXQOAzxAN+2zkiM+YzEf/9X3xkJPieBxvZPS9WCwojvle75/QVWt+50MvNbHNzS6aXMs1upyWWb5vYmbcCT1DuIQlNKwPKtrE95kTkddlc5ueiwD2agauUXauWn2uup+ybNMUPNwqNsHCA4tIqufICGKF35Kl0RZTdC1Y98ahTIUPFDrHjuuu5IHF64axDkHPETsIOxObFqkbhqUtQ99M2vjsa228mSTBl8dzol3vxMzgGhAH8BFtG8Rln5UOvKEKc5b13pBxmGjaCLY42+6ppc+2ZSCnOmVYktML+rnIvFSXwts8T2gYdNxg1LGhqAr/LWbuJR/6sJnuSH/0+gjaucV3BEv4C7wq9t9W6r3vWsGC1ClqS2zekDsF3OCLClv6Zw0JP5S+fB0Eyd02fdBwFn8tTwgv/V72BFIXPQuZr/BRQfhASGaNvOxb74CEQICANWhacVP3WNylAVWAFyYpsVcfy3k4WLmDAt+MFDJlv5X4Z0nEZh4eUrKQvNz/H3iBMwTNlDM0Kfywh175HKMtdQm4/DTagysaySD6Q7cIwpp12MJR4mzFF2JD8Vni9ZJHHQ0pcX3PNOgmAcG8+WM7+JCkOY28pZRX10ssipWSNF4UUyaAh9A0lrCM8aSIDjqrCAlr0qN6cD7L7rx1vT4ZzLZY7pMHJYRByBn7H7SIW/jAw2Q06nkaCDnOjfQBEUk11FTscSsKlICbfcY3du9ci+o7AVdqaWipvb3VTn/L45w9ppgt1BP04D//ta7quGdALna9T+kxaV1DwQWRvLiuuYs4+NXWzwCgHtnBLOoMwMO/QnZfMYP7xo0yMsTsT6iEglmtNXJdDL/Ajduui6BXkZswR1jp5ibUIgl/DLtkZt24geHakn6Jj8E0SB2OG0KPDiB04Swm+dXd5G1NyoqUR7V6h0UmVRMcbQ9lEa+KSTcwHRYbLrYAPXmCkHOxjRiNvqUVmc49eL6jASMn/EAIPYKhpOb5FYkVrRCe1q+8m+UiwhI0atuqDiGmqbW9NLmN/2kKh9LJXSJ9nrSI7NUcM/ekunQ9M3Wi/C7VynqlTvcDqFB2XrLZYDabbxcY2AVH1yUfd4J9vxZ2tAyiDwSKVslm2nvzI8up0m8FsYLz0l+gQlAClK4J6ubu+ogrvpgTThgcIRYii2J2HPdlSZR8AJvFUCyIoMnH6gqnTQcQ83+vGSDANPGFC8ebIiJTJq6xSoe6zxWCBlvuWVcXr1lpZ5yNjOtrK+pnWUVzBF+ESSzprW0nihTErTFi7dMcSm2sMBk5saWV8kY687c3iqerqwwiBFDu20yhlGBXmr3UV45fGJgXwgjqBTIYr0dnfK967PwW7XF5m+OPEyByGxcnvQYGIVTSFm2Y1pqL4f7X1BLENNsdpUkxa6dS5eETsVKKKsUByw1LZ5qtPyOm6koaKxWRhVG2VBZe3my7wYvfCiZm8GVypbLY39XAUJXQwZbDsN3tN29eqscgLCoGBUpi18INHI5h9Rwh60vGNSwHV1jGXIpcZb0hVx7uxJ1eNboNnhRtO2lvbrBUI+Zd9hZzuhctTAnQGko8H3GWPWZVSgi/bT5REIjShVXCqBlXpyw0KuAeNZRDUgpZZE9HE8FqOoPUSeSS1HAwTyQagdBw7gxUWVN0Gdgtp9XOHCEf9oS9F6LOoUVWsIlCDFgFuzyR2dN3ouLtphxrG8Kuy0PmXBfbVPS9TwgakjaLk48vjGYxkGycrpnGdfvWJXtQXVbHEUBCDkg6CLk8kdn4FHPXVYvNdvxE9T0Il4TR592dmTpFjJrDXg5pq2Bt+dS38UDrfbOpHSeItL1R+kQcV6TqTcFPphmBjlOi22nPnHbltxGK1gkGq3af2aBPKf4yD5ivhnRekbIdA7vCcKggZl6m6QNNbit85oZ1z03hd5YZym22d4YL1x1j0OGmYovKPfRO1RdvdGpKHOW6cVH0XwUbTc5bqxLOVswVsgcIVJ9sGI4ixGE08kVkkGnDNIXDYtTau7RjhMumPduZUjqJfJ3tkpBN3Z7Qk64l+mbbkFXSJabOkRkLsur06jJpqSFt7EyQs8dsuvLi+YG3L264HbG4gtEp/EOMBLZ0e+sHtCBafXsaC5sBcZ3RPyA1654kewC67rCD4KA+I2pv0SUSL8AGRY2dblKy3eJzsnbUSFp7m2iuBv4sdMg/DxfHzxxdqHBho0LjcOqFNa9YBl7RGfwubJ9SgiNw6+McLa1sAB098G9RKVYQnx7lJum+JVkhamoQHJoLr0BLnSxCD3FYzQlR/NdaNszQZ0dG1+O0+1rnR6z0sXwIWouTNcaAMca7YtUD07PzoSPz4xVnDVSBQalOs0t3GKI7oIwYPpqwxWPR2p1V+5K3kV4AmEAjuYHFzUQmpWxs1DHIgtOtstIIw+yUGxavybbLT8u2vv9XYasztHQrpyjeqEeYt6IFVM/QYg1uPAmLTJwWErfE6zkClF6UO/eOAEvaVhgPkgddid+xQjsYBEcZRec/OhaDK1fvFBwvIza8QvkP3prjkQQA8JFrlAZ2XPnBRJUwdgMhmFcoM9PjUFfTIvLDO0L0T+ehb4AlmIPngB0hnaQEM/6ll0GpkyS1BRwOelXjeMo470sNqnb5gvMQNRZo2jm06g9CbA+oJjKWcVCdlrC7WVOIJs5+zcot+NpLJ8EohLqgcuyg9mWbYQcFH+zSXpdyO0gfz1WAx/caHW0fMKBlTB+pmLV89mJQUoel7qSc2MnxrDGLfOIZoh/gQgNgDrATCx/1OGBQV+k3vH+uoTP3UhQlezTFf40geD6kqE6M2G9XEhw1rNMf91qc+9Lql+qmqqt6ximO405VKqKZCjtd9e9cOk7P0iUmM5UaAAdrDXbqGSPQbErM61QS8opFXfAoYRgMRmje1D0o63wIRzhQbThrlgqq0GTBLpX6aJ8P92mG6/2FzRx9rkgEQS/l8RR/5T0Uf+MxpYHySV9K9JJqtfNa1Uf/ubqUnoLEkiu3p0qngb9YEfnJqUst6gnpzB1oHS6pZniTPSdJPe8W+nppf0Vj04mnDSvwPt14RUj1I9m0HtmuahxGLKKo3A9gq8OUEracWN64rqjwCV5xa8AgIyJPO9rS7IoXOKUAklusveWM5GwzXhad/85LT8WjtzNRMpy4tsQeTM+u4/vlt/F3jf/fbdq+/eeRYeFLzszxZOAmMrRXgtZ1DoEd2Z1egd74HDHNd8xMhdFevIcugsrlEZ2SMOW3FKpMW2YrrskwuOWkYfzL/cnugZp2X2aPlHQ7PAIoPTIC9gn73u6I6y5oajTpsRzNTzwiL53vsUf6YgVsdgKCcaHKWRN/+eeIvGd6JNEeiaY/Ed5zFATQYPxeLljQmfeF34udR3U+Lp2ixMMiOLI8LGybnK0IG6mX1yXiUHZ4sd47KK70JgOKlyZVONh/VaeTpvB3qP6XaQH8d4b2TdwQPs06jAHF1XrGrcOApup5H3uAjnlzYoRjMksHN0tuUUEZWkrb0CNfzZxo+vicTPMEhB5CXszZoUxQBDfRJ9kY5avmzZ1N5p8SYD2QgIGilHMglcMACMlcvRbdgFiytlIYSJPxqBjt48LD5x1PUyPKNJj3pSiRw+7t4MavnoJ7+i+fccYMDhfPTDAnp85DRXMAUNTHaXcBG3KRe00biT8WQDi2a3k3o0ijQze+BejQRevzId2LJm2o9Wy71ZQHp9KYpqLuJW4d746mp98o6LV/VLWGusWnWlxnmVs/8CjeATk1voc85A7OeXGFDZijw0ZMZDIIwnMcbF+Ef6alnAWVRw0GeTqHwmlnObk9S9g9s3B9cqpS8haE1hcHHjjUqP3SB9RDINerlBySrv06xJhhrCoPqoN0aLKns+ASisE6CKohkjg7rcNtrAYiswyuVVFt/DYwvxx2rnirRIcdrU5h1rIBNY9z9q7r//i1MuWi3vBzRz2+Ey3zvqO+7e56T68pxDQ/7Pnp2rKH2LYOXD9KnaMOs0XKchLT9GVma8E/6y/CCzJLf6/JdABzmDd3VNiIz54JlhirXR4oRh10omVFm9Xj7ra/GM0uBQ1HZGLmaUL3LuhTJQTASmrkji+jI0uhJ51uMNvSf+89FYUHAfe8UzPHuC12bmaUmRfup9nYl86+PZagBCtyd0bJF4NLdp22PepyTVxX/GvDKDOQJ25gm8bb6Dr80+wAPcmSDwNtmmurEI0CXBAGUd8BsRCc5OB7uOhOKxNkXKG5eOtkZe4So1BTsSjC3aatobdxnNLOfbZdgiGKJN3z3Te+LAMUBcAyQCYq6gOsXLm3x9r582PuMN5SBJAqolzo0oTXLbmIiyRbsf4p3jiDifKug8Gn2Gbj9xsAf8ps6tXojuhon0hIh8E8i9JdQOIN/g7rNYb/nZHJPfDO39oILUV72lHzX0JJsBzeIy5uYDHUx9MTlq0B10k8q78+JaCD8a0q7xYwCjuFRiPbPrBpBMRiz0E9uuJPw78izrDFMJKfZtuktePrtrmoMGb9XOwbZb8GtYqv6PPIK8hsrowHIAUZy75D7oWeehMBzm1FzmRj6t290F0Sx9elVypiAZAu5Bkgmc9/Q/aLM4CafOzKRC32AnMD1LlrN/n8vYz2GXyhHw7DL3+5nHfF25W2EKrYu6mwVdkOxKutml+u0WCR/aHK60g8rzolB9DwhPxpeQpdH2G1J8XDQURGg9mkm/7kpXhjZxN4nRuNokMprEHBzVW3qH3KBDZ9duHQrylemLhrXAPqWooToZNg00dzxqmjl+UK+UnLzXJaJsMyQMbnbfsHwYTdcv68bh3Z5hFXrOVhmxKGXGEQkYKUUXbytdEsg/B2zQbXkhyPtk86DEt67R0jHdY4SxtAlLC2g/OhIl5kIiRPCrq8JGKHGTbsoBAABRymxbN1jMhi7aiNO2ze4GKDUGeVZfy1JdA66uUqGoDN65IzqmwCMScahDJzB95nZuHlUgQKeIBYvniFEIVI9NUKoJjl3TRS4GL8s0bdatQlo/UYW0Cbt2DuVoyWfvEN4BSBf0rfMl/LZwUAHwDPzNg6uyW9D4a2eN7LTyuWnukSjH2cpQ1ARCrgvwTDCIBijjzUhcOXsD8iAyG9B7x+6PT4DN0DNaoxq1yS7VGMlmeCSbzkg2O0aiv+FiUfl8aSLIKZtHAQvI2ispFWyShirx4BFCkLJLCxCUMbnJLAuzGrCMtmSsLzXVBprI18iBqoj7bSzh1e1ydkMg3IEhsP5FCUpyNxtG29DMxu0qtKJt5cvm/bi56IRZ6EK0qrkNNXssneYyQIn2PKskQuY9jG5ezoDZ2V2LlSqFx4PSEUeuGL4x7RRTHiTD51V5uloBANqVVbmscGM0GSBMaVbQgqOt+GOlLikOaQYEMK2vHlFuMI9rLTp3A2N2aoXcFWK3Y50wu7EiSa/fkIfSp8sZ0XwtlyjSAnmX3RzHoNq1xMR7wl9iFbSoiL+/PCO4Diq4jpsmisXm1hv1P7VZgxvGazZietLW5u+Jd3odmLMhD9Ass+Zxb57/+0i8e3+Kma3kNV4LwhdhbsXpm+N6z67+XwTXggQ0jippTn7GN1ovpMv5UXclT6ayP0yF9Y/4HzFaRRdX3Vp3CmrTq0wjbImthTeLuvrJzBS0am+TIQ+ZLigtD82i4VlVgVn2yUj0XOIMC/jRcu6eLkCb7JqHB8OOXMPKAmw6hvYm7Quou0Q0bLutVCetm6ihmNFKerW7W4924DadX7V7v+rr+6ql4dDUzJt5tZTIV/zqILnwirM6Nr863UgSjzlflAQeG4wwbxWhJf76q/SDXH3ojyDhBsKrAf0DRnt+6SFYhVf444a5APeksZ5fOLBQBZ7C7bTrezRyX3Ws5mnUxGHBVyW0USedMs5A2h9c8AbQ6QWlNSA72xTTdkS57XCrlf0NBQtSqUkEmCqfio7b8L3Qn7I2MqouMvRIm5lScjm70Nb7yYDfoqqlWqzqUZtQqy6hULcqMYtu0yxmB9FVmsRizO1340d6W/Q/lCD65VMOz0xltgij6dm5xb8oR3ha4gs6bFGvd01AcFEVDQlTphtI2NFzjLnKOR7P8/N5GOqQPwVEUzxOb0QR4anoLraq9UHaiS1j++7HLIR9RtLs9EWb/fgUPYfr1AcphE1jwMgNFL8melRSPkH3OkytlkqCx5qp8+u0p33KXWWdYl60xiW0VQxUEGZTrh6m0+t0RMA55XW6lNvppWZpi5KihBYb/YIE9inL7S2tALTzMF95LOTYUNO5xeG5jWJ0ZdHlWTWZ3NxAJqnPNpk0EO8GOryLsG6cAWK06Wt04zhDpOs2dHZzM50dbPE21K6f6m1uQfVuInrGsM6tiggwLJAgoHvR3wbcsibx3HSLNJD6W+C0gRznRH15sPGiezFldf/2LUgA3g3Zk8yuYpYrRSAqmW3ToBS3QPYvOrfJ+9FjiBy0rGkNnWwYF+07iVq6YrZGv7ZSO8NlnMAWSIzR1NdBmyLMYkcA1mJDJ/SxU5AXogijZ+0GfTEEv47p/bajaayPjtGjxGb50MUwrTgVLEnv/g2XKZyvZbFKgvogs7lzRrqwNVJ3IPFI3OHBakaHQCVKnvj3MKWMiVCCAHxUf3l55h2fvDh99v7keEQ+diT7EagV0XRCgHfdDfMlqwSebuC8nJg1EyMfSbnt7js2ioM0enz3/s3bk2PaOhdvEcbB2rblrtOHKN2phI6WtnewDgUieutQqrlvvTPqlL5WyNLFuwXeplKvAJ9Yw6dG5GAmQZOeq4VEHKrPaUhgIeqkRqBUvOFDA62bNUoZbfCSixQ+ZbX/9YGP1wGhSYk8t2WsjZyVMVdlojJulC4wqSiauvBKi5Aso2Ve+hn8tNEcCjOQBV+88fKZ44rfY9BwD8f7j8eidf9F10Ab3MY6G/ZWG9R+BlWbfsWI99pQt3ymsVqUZgmPuV1dJqPzaDZp4HpzfK7BariFCTqfOtVbVsrh7dgd12+1kaE6eNIjZ5pF6bDBSDw5MNrQhwrwth6nSmuku2WNnvp96vSKe+hGtc6SmB3tFahQpkHc+k8wuc/Wba1rTQvFfVwo0/rVZ0/Ybf8YanDYPoEUsWmfQGK2ywARk5X9y0GmR1cO/omaMgz//x9FGQY7rCcPqKfmsJoaJY6orS2SQbtST/GpVkz7ffoGllu1+ks11Z3JWqODX6jKdSO21aRht3tiG+spKyvB3uDWmeTmHCGIBBv4a1Y5t+ZJlsllGCSAK5mn6HGnaC2Rxnhxx9dLpANyZnB7KbMpXU6nKNHs10QHXjCaIhI7PYoldBVUiucd1EsSiWAV9HHU38KCbrSLsCfTerv/s/A3lAoYaKMLEgvF7AW+tvRS/VcvT09A5Hh98g5NdTXnVXTuSJzXgTQUYarNlkcqz7OOOOFC/daFpiPAxYvk6mYIUKgV8jPpckMtweCvMIiQDdDU3FOzORBk52VE7XGJwYbIgJ3pfIPY0mFzflSArzccbGXtpxwZWfMnF2+0q5vRHwaXhr1jgQ5j9kiyCWilHuGATrgAzujX0+dCf751ex/lTEmLLl6CZ2EjFGaZ0LVjAr7fvjFASncxu6LGWoN78fxvt2inor/uEwSDM3qkZI+rEhb8Ni3ojX+CG/8KIy0qcLj1PIAnYBNPH0MTf/CJd5mr6NNc2Kdvjp1bN6ZkcJXN3HrGbMlARcsxgrmUqNSDWX/ghwZeyXwu4wGs6oXgA6OdmhZFTET6W1Gu/0qeJkTAebzVkQs4Eho0XWxRldvd3jIE+Dp8PKaN6jRXfR1ohEesV/QP9uhpVFSL+bnWMBTJ1gmFv9SbZHiQekwX+vjz4xE7H6eYf8Zil6bWZqpTz3SYuJ0sY77G+zHP1Umz56cnr49P3sL89kmV299f+PMiybb7dJCjTOklBVyM6edZl+f3ec617Imomm6Jhe3vcwch7i4tDzzACrFbttMqfER+rdYPnipNAC/p4gdutlIG8AM/OH3NVT5IXnv9RmkBWLl65oZrXQA/qqfepqs9wylO9IkCfqVOFfDy7dckmkY7dOwSSqrgk7rFdkKQugAZOep+ehqj3OEosAlP/STXmt3bM0GWpSCMRq2CZ+qx8Nm/f5lWTLpKclIF2ujh1OIZpWykQjXHJyladdT6VIfyGNoIXsvSvLEwzfhud5Vh0D0jiQrAHCcQAGhNzY8vz07ovcwy8/2798dvfn9/UzgpSr4qGcCsXFBWRuO+RbRhoR+0IWjjgFGWTTOXR9ORda1nlDZWLDJ/LSmsgOqounygii0w1gv00faoCWsyZYHoi9qInTVaFPY/gh8ca0St9RrC11UUhKIg9KwOek4rXtFvxK1r2XhYjs7KuYd4gyWmNnHxAqTUnjh4Wu4h3a3bQ9J7DMR0w0Pcsj/ynTwqyy4tTqbSyHbjXhT1o6nskEfM5R4WNHSpnqXXHdnN8O46Wmnr3xCw1G6p78oZjL1vA8M5kkenvXfdEapEzPal3I704cNmUuZ17gxeER/iPXGBTjAGbei1GD7/ofCAEgMPJ54yAKd9IhJwhc5BNowQxRWeiAU9x2JGQ8DQ0P+hGmsmmMYFadN4IOqBWzvfP8QjOd1kO/x5b2/PO3n968vXJ3pD9vaQzQO98mS8xLvboOA0zQDQ1fIto2TmR0JV21PD0q10UhzUzZ/U7VXOATUY3dj/BfWxyow=", "blender_rig.py": "...", "app.py": "eNrVfduOG0mW2Ht9RTiFWSarWVlkXXShmhpXS2pJRqu7LKl7FuYUOEFmkExVMjOVF1aViALWD17AgHcGNgwssLAxO4BhYzFoYNf7YGPnwYD0J/0Dnk/wOSciMiMvZLHUPTbcgymRycgTESfOPU6csCzrRZCKeMrfswsxZvazmLte2Gau8JlI0gy+9OFLkMYhPQvCVIzD8JyFbBIuQhbxmRfwnSgb+96Ew7MA/5/GPBEBZ5zBp6VI4FWWBSzNAgABnx/7YeZOfR4Lx7KsHW8RhXHKZn441p/fJmGgP4eJ/hTzwA0X+TehPyViEos0b5Zk4ygOJyLJn6TzWMDEgln+wFuInbxjmjTjCXza2ZnG4YI5TP0Gs5l6M/VQBDBboX+aiXQkn+zsnHz94uVrNmBD6zFfAEZiq8Osx2EcC/r0mvuZy/XHVH76gnu+/PRKRGES4qeXPAgT/PCcj/HHs53TV09fP32DsFc7DP67A2uwGMeiz+yIJ2HSYeEkjYXosAkHtHdYKi7TLOZth/W6jJrAYi0EwAUYixBWMWRvsngcEjiLPloA7bjDDu8fddjdLvzXYQfdo/vtjmzzRKTc97lL7e532HHvoMN6x/WGr3jkyVYHAO7g+G6HHdVbnQAGsM1RV3V5v97mJb/0FtTquFvp8Kj74C40u955+ucnL0+/evp69OTFK5ibtY+kB7S6/zwLrjIeHD7ZO9jnSQKksS8u+SLyxQiAzkRi7ezsuGLKRlPPF4nttvvUaxi7IqZVlHga8QDau6Ez88e4KOrpu4y7SeOz6fjSeAYcVWkVe7NyG2fsi8A1WsjV897nvdLA8v90M+BNnnqT0AnHb42388eL1DceA8HGIvHC4NB1EvylGegk9MO4aOxE/lWtaeSJ9zyffCSAvPQXHk88oBL9FeY68oJp6CAvAyHj61OkwSG8EvF07rwNvcB2O2zaZtMwZlPmBWoJvCnTjcSll6SJXX+nnYP8DGBOCxAJMKdwbRQnDv6pvWvtWu1225wY9DdF2YavA0AQM/RtChzvJhdeOrdty/HDmdXR3YK8yWJsrCkpyRYLHl/ZOGVFThwmSxgASWFb/G2WpAKZe3UtO/dBdBBC8qHYUyB8wCEKSfj74qTPdndXCGPYIvYeofRonfU715Ldd3fZD3/5H2A0bjbxXJS4Fk6m6LV4y2oz4SeCWfDhs7zHqaU6oOayk1aHtX7eapudfPgfWq6wFZdN4Xurfc2iS/zRMgCK5F0mfJGGzAAM1ICtbePRPEPqgc668IP80mFEUQzkawmk8VakXmrtAbw29m38mIhZBioiGaUhSCw1j2uWGGQ8tb4M4wXvs1VrkfmptwTy4q0S0swfFNJaPaCvNGxdl4ZlN3U9RfDUK43ujUIbrmVpSq3Eq3SrMDzyim4D7LKxG9W46MgE/groVsT90gLQI424EkJ++M0/sDeeAK2W9BkNn62fWIHT0mSYTYqkXZ0SPsxnA3MxCWnDrPJeqjP7wqd5NL489vM5Nr+8sltLzxVAYC32GUvS2G6AIlu02sAl0CxhtgCRIn9kkQ/6mbANcypNtPZ6PmWrxGtEmNuQ7O6uWiMpb0pMTWppAiIaxImSNbk8cXgUARps66RoBJQHk0GhLGUgCZQSlDMlDyclkTXxeeJNwbDDVlZbD2RSdJl6EdkVK2ueLQA3MHnQ2aAXYhD6HHucgGKMs0iQWSD7JBvwCGyTlCdVBUNKZsLHoGWwvfzE9tkYhCdZSKDwQLTgb+rTdQ4ARNWMRjM8J21wjuLcRpUVB5ysqnHM30v7agIiyieAchi0nBOa9rkS8TWUTtEQEpMUdLO0fkF0EgLotYmieHjQandY6Wv7GpazPlH87zMU+sxeoqG9wuWXSyTnAhKO5LmamZbfciWyZWmxsmWxQtmyv24K334H8iRbDkG0oCa5bvdzhuRgNS/FZM5BkVCTSTgWMYmkaJJCW/azMkeV/ptaaex9/F0wy3wcapJ6YFbDJ4KUfy1B8oIlwPfyVsX3vFkxp2KmZGuNSDWtJX+tSIF5iQJpKD7YNh43dKoBqaxUpSMD5hC7AuXqe2Ip1s3bmodxIAi+1Noh6DpwAXxm/6pqMP6KFFzKP/4DZ7uPcVxhsttumqMfujfM0PrqmyfwZCbisMbgO+WFWZ2jElnmEwSiIv7osCWyiBQHRY9njpeKRWJrI6k0sAUHn2IUZ0CPYHC464d3+sWrPsvbEUEskLKuwBlBGThgXWYHIRvHaDVKhpK/wJp9G3jp1f63AbhtfiN+1DDk0qwfxCvZoM+wPa5PgEoMbH42QY8QndpErppigqa+iCjBnQTt5PMNFPflx+8T9IIBILZMlP4dtkoAJKEtP/6dfEatP/5TAC/uT2CQ+yBGZ+CKrqM23wOeLKgNbCUp3gCvUrqBXAVeBhMM5gqKi2RHJoKmiUmj66ZpPadWelb5pIyXkVNtxKExD+x6zEFfToSMCwBQcBB442qSUYerCRoc57ZhNKe5gTjhyJ8Ruh3cz4dVBSXRHWVBinPwph+/n6DPtxa/NgoqMERnPEZOYWkIZj0MX/hkmyIakixhICc9dNXXTsYjVxQ1TK/NHrHe1hMCN0Our6BOFZzy9NRDwjsI7UvgqY+/A5oWMb5qoLig3cQLRvgyoAYJBJGseX/z0Pqs1+2CvJZssiCpmnhAUgxZNbf112oFu8oE9ZHgPPTPLH+qp4Gi6kKPVRlAYJyTHh+eraeUH/7mb//3//wNW11cK0DKZ7N+GfwyUIKSXmmDD/fqxTOKsVgnmbQueCbtjedk1+CTwsLB2E7scZIX/ZKJU1gz1uOy6ZIbM/j2N1UjBgNDMMMcnfhT4IEuDdDAkS7m0hMXIh7N04Vvu4OvwW7rsLnwZvN0cBfjIvi7enzhxWLwJQdzocNApEZcPV8kM/qkkGZZ1lMZ5GOHT5iNUTIB7nobSS8DfE9jvgByFzEYtYaoBC0RAVQKCMYiAWcJoxUYyVNUh76zC3qQPtzgyFsqzOjgvMC0KZZTLRes5Oeut2RojCaD1jKag1WeXvli0JKz76/kv9fR5UPXS8BGv+pPfXH5kIN2DPZIl/UnAuOcD+tUaqFb7k2v9lTwSLdECHsu4HGSgvHbB2MgWwQPZzzq97rQ0ZhPzmdxCKZ7/07voHfYu98Ee0zhjD2ML2ZJv3cEL5JV0b/zgD9wOX84hU73pnzh+Vf95CqBoe5lXutRAyjCgZo3vZV470X/6G502Xr0x9/+t7/6fB8aPMJWj9AWsGGpcQWs19kYg6+c/FdQElHmJ5w1dTDW4NUQxX3xQIjWo2cCJDePkURQjr84AfBPNaF+vj9+hAEFeF8OgP5qzsWoaRb7oLNg0eMkD52+y8JUUJN3wHRgOQ5ASKQ2RmUd/GO32S6KnW77OicqJO+CON5h1Gdq/dlysMIfimZI+pVm1p8RP/TyNsQStUaSURB5NDybvsuJJPGExrkvQ8UjHnn7GDocrKrk3DLJGRy5n6/eqbEZ5KwZK54MWiv4e51T9IXnpnOgsO7PHtaJW1JTv1uhtCYqq5NnC8S1H14A8WS+n4D1I4KH7DLeS2D4oD/30hheAYnTqgCXrxUvwQrL4VeIdFqiULSm9oj/+jEO/yEoUTCK99Iw6h8hyX7OGUibaT7/FH4X6aAFTjwPzlsVArXKlHl/fH88nT4s2KB3iDBPwJSMUXZFPKCANRqVkY+i6oe//OvP93lBnTs7d9jej/iPzRRToN4mQkkA4ksvASOW4iTAa2iC+ehVRSwEJ9qTfm8em+6zaRZI9xv0KY4TVf6Eu2i3EDhbKtu9R7mHBh9zFdF22GO0p5G3XbGEh0s0TQqJDMwOb6sdmjtgV7AFmGwpOno+QIxpnwLgcTkBh6EGgp+DZOFh/BcxufSWIQs8aI6MS2FpgO/s/ItvviB1qXXT6IJ7qZ2kHRYAaYCXE4MVI5LEVDSRwK4JLaA4oKMxfxsysNPIlsCx0jjYrxDEr9gixLFi1AVa8yUakrmOuZgD81FXOmqbpEMLv4P3kodv8ZkL+LXOCl7XA7Nh9nbXefCgQ82m1hn47KDhJgNwl+BJa4FGCUYXTaG0h41baReMSqc7xSBY2wiFQLvEFyICuMe5BYbQwSwzxxBzD0ThLHaexnEYo+OAPAYWLAKHttAzkaiMK0ehjWRcYDKZhxcjPY9RiAFvWNvEQxeLox1FiwekJjfzcOG5uwA1WKhoKYW9ANZkksqtj3PcQamCViYiNXPA6AvQ5hE2DP2LLE3DwJn43uS8jcIdcA80lZiiblWH12dDnAygFkP85zKiUVARrpY0cNR074AnKOeHFh/pICJJIpeleN9H4pgIP/N5rPwdoC8XLdQAvMkF0Cw3h2TaUGgMWU+J5BQ3uMAEb0KkuCT9+Duk/w+/P0Qi0FaS2g/68Ic+wyDHGFRqsN6XULLrRO6LADuehv7H72chbsUpVgxym7pYcWUo0SaHxEIEa1PfADHnFTVsmURt4gT9FMQrykp8/Igdg0lPuEdrUHUM1kc0orHYoLgD4GXUIh3miyl8JjEOzDIXQv8zIr0DPwGel2I0nnUIRrElWUXLlIOBTzuU0FIImAUy+Eh+nGWei0xOXWnLFr6EKU8FWreuCGsQ6Sk4EgA1oQ0+BA8ISfnYF3ovFEHKOACQYP4ZrGF+LlS8tgbYpS1PMXgTZ9gSFjoeWLnsZnb88Xe41dm2SNYtonRgWdTfKMY9UK6s7ypYzQsDYKFTLYrahpQkcuwBefEAnAdDTl4VQj53w2RwC71DWnJh2uA5Zgu5g9+AkuQuuoP/oNGFG6+7u4c9EG49SVMimEGzYnvbzoUZ0QXzEiIboi4iBP2k6Eo5AbbEDRmiVtsBivci2zDxG6RhxWBFXpzEnnrk0pdo4n38x8AxZG8u1btOt6fkOLhogtQHIIp2fQPlwSBSgS7CH/7iv5riO74qj0vOdYDYcLD9KA3l7rGalCThAoC4nAiY6lP6ByUwB32+capT62uSWFGGAWUYbbx2rKAcxLXVvA5FH+aQwQ/y0hEtj11n2rb0o+kFBfath8ShUimcNDwXwWguLu27ygROyT2egggHJOPeMvqop9JbD1xyYWEJ4KF0NUnp5V9IDfdZjSmUzga1cIatPOOVtAufDeUr4/xoeAxhqGc41jSPD4S+i6rKBy6wsUl72N876Bo6Fx86URjZ0LJDXYC8lYw+JfIBmQd8btCmMgswaYJf2uobyF0/5Ck0NjaQ8beFbFnADIMRqGOgFmkQuRXIOFV8wy09VRaMjmHgV2OYF2F8bnJPjWQRREwmkKSBOAs+WZRvUmmqM1PUa9lObpy4bGsBn0v1G+FhBIaoksQ//GmvUwKG1L95lBu0wo0vK00xkGuuvtFOUaFPpBpFlrgRHG2GDhBDhLs2+3wATm5hLA/ww81gpMYajMMQbJhChQG6NM0N9IebgWk9R9DUl/bNrymFiPI8TGaWNHRjmx6jnOdxKvMmrDfYJM89mEsNam2BrEKV0tiK7+XUjRI35QygHt1GOOvMMPCDBXJLrROy4ikMsEqvImGLtjMaIYuORtdSPtdeIWD0Ug7Wke4hWHkTuzyRKJakkb9W/DzF/SS/gdulewPw0U6RkiLPdXPe0CdbudYoPUAOceDwgMwatU5qFIUHZ0l71HDipARS+C0Qm0c0pIgBqdwpBylVTNKae0QhhmUr+6C8FMoX0vQx8fmVpeKXZHptphMVslFW0z7Z6vm+HwJHbzj0MzIYrApdgx7OIhfkip2AQz1JhTtQU680xH37hi5Q2fDdXWZvdhHBTz+FhdWmiHLl0fgo90KLjxZF27THVXObkLud9XjQL0xFsB9Uyh+G7hSw3EwknU6aEeUudGF4rXXromqovVGWKbhdIpBZn274kKmeuQ6RaDvNIDC917Y9iTWSFYCxOmUaOJAL9EZv5t284mlz02LNyyBvseqVENZpGHiCwhn5nskVM3IwpDlqLD0x/W0W/rBfwO6YoLGjeDIHxzn5f772I5zXaFRb+9gU35IaOrTjgx6LIokOfsJnqsWSTNYAVsa2NyUVqoxAaIKrB6Bst6JEtF1dUl3SzLCkG23NvDg8vNu18jxBelzy1tvaskRw79FrdzEy7rz3Iol0SjeFx/bwfXTW4LK/j5QgHJ5hTD1PT13PC9BlE1WrKHSVqs0AS6f8badiYnRK4lqiQqHAWURHAFnOpso2P/ynf8N2d19KoYPoRgq5IYq2u0tbcjjjUhplGTboLQ/Mi3z4OMRYIDZGC1e9wEaYcDPikzQD3qFnP0Gwl0cye452vLIk47EX7rx5cfrN69G3L+rZT6frsp9Obsh+MhKfHm9IfPpGJz59+2JEo8AhLPvsvJTVocenkznyIFsNQ5LNUR7Ya3O/0IVfXSu3ESFYHWM/Fqx1nZWkqDTvHX9HvtdIycNclaUrBlEfoIxKXGLoesAsZsv1kHuSYFUnoNziWEyksJMZU9vNpNitzUGOYA3zTNnKxs2dO3fYh//1mvtgTrAxSPOf/zJ4mmtYeMOl/DD4MU8Pu4Y/NPJrh30HXgybYMwefOZU9KsaAlX0H3/7739NuQZxeAWgZPQWU7VAhgPCyKKJRIr7HD4TKIydAqU451E4zfPZSw6imgZmYzvgz4A4BB+zvh1bTtvWKUBV+7kGNo/hAgaXuHtGWgtzG1MOjgkQJaBEwBdgHTU6vVG8XsucxiAw4lBpEwaYGMd0kkQlz5vbjPmJDTPqqz47ekQGhtYPT0WrZD5kGCTiLeqg8u52eF6MumizKq8nWfoySpKB7U8mWIhEvLubiAXGhNGOOFliqA9o+Ye/+Cd24gue4pBhfGS7wRdoTk7U62fQfG2ihZVkY48CSLu7Iom4T6lWkYhJSsObFM1+iWkilNmcfPiDUxU/2hYqhoxh9A+/17aPDHryD3/oMGlQ5mE6/LIAJMAAsvfQ7zQkc9dLwlonpRyH1x6FuGB2KuYO8rFD7Pzx+xkyFY9jb0x7U4B3XDMa1rvMkwb1hsi79eH3+S61cnMpdA9DR1QJZY/T8HHPMATSe7N/UkcKCjsY6lPMtctHKyWOsaO+u0vHfWbImbrb3V0T2jWJnILqrHJaCsiXE2R0F4N9utU1W7VAJPxVy9wvwSe/boFAQY2pqK+T0zoeAUlQrOaOxNS7HJHHChxAsjXztjApc01HczXm3kdJp1QiTB9TzMQcHFaSSHrqtXSQPyGXw+CA+bQylBpHT9PQzMrAlSalIQn0MDUCU8Enc4kwbCRjUcXwKfeIwgAnWmf01VAp905MKKE1EcAtgU4RhrGAJtA+Aa5iHaBCWFVmyTFonUtzvZ2ibgrnTC1NwH2WD1BhKBAXlfC/EwvoFwdDsbmcdHy+GIOYmQLx9YsY/FQF4Bft9YbwT2DL3taGLr8tkQ5jKmQEULGYzEOnZosCQtqb+GvM08ncltawxJCKU27kMcUY9Np65qDdkCWwIU9IVOkd5zV7NIBgGIMf4o8YUh+e5SFyYAZyhkSQAbshiqnrdsPeuAd2pw/GgWyQ74t/GVIeIqCmd72/KhpcmzsxsBLIHJQSCdoDsBVMBNJEksZqYaYORs3Wh7FjI3yN8Aipg6b4MOF5QH/L/hziQUfRwWdEujsrtwAcGamC4LTkqTxjOrwKngp+a1/TAav6j/GwBVBbZ6XZbxFmLPf7n//d2n7NLR/FLnJ1S/mL4UxT4eJqRFIosQsh/oz7wLmUkawl1Axkukv5pTbZAWgGLT0Uu6QJoeU0xgy4diHB0X8gesppyb35nJ08OOt88+0bPJ6pDt3hBgMmKMjgXru2X6gheIl00suowziDF2QF5XiL2Vb+P7ryMNWRnKoTBTOLfFp8RsFLetRutCW2OopYcvx1QvBiVgkwIx712tvwc4fVlt5tD/u94zMQuJEPKtW2MFBisdwOV4RAoIb9u90ztfhoy8vlt8Uy7aMAeU0C9Am4mQqNgFJaRvenXMKdNbjSK0hBEh5c2TcjMV+vG/cGchWw3bqq8zwYK0IcDAFDjgfGy+XZTgPt3TpPVZ86n2ep5xfbefTVmYTRVRmW/gJDoSUfkYYajdq1PnDX0LRZVtd1r0799FN7dRFPkv8/Q2x/2ijZdpGxH2enVINg1cBXCfotgmAoJ+YYkAPIinzfo0C4rRAgXDcLAq2nQDYd57IJt2CyeCJyxVQmYMk9UVLiHr0ZF8aTef5wuYDRyobO0otxdiOyyq6MnTTUCPSaM8lc7sDvI7SSEAs2bd0aP3rJiC+xCsPYx7gkrQbmw3QNjd5kZZW9tpVCkasyVWzhRO1rVNKvTl6CFl8unCwRLthUmGfT7Tu96TV8qbuuAArayhOieWMMkD77Ao9vLRzw5jE7/fpndAT12em34KJxl/ebYdmzYe+M7bHZsAsUVul9RT9We2k83Te18pRSOpQCnsZcwLzQABQOfcuNoEYD6LIWLcK9YnGJRDl6/uL1GwqhpjoLZBZl+uMy5gv92fg4KVpgYip9zsObcsdf01qdvJBA8ORkisIjQlen63TrdIlStSgkQraoFQCze3wvWXhsbw/chvhqDwY7QNDee47zdeB7R9IkLbr+TIva2ZwniFNBwzyLBYGJQhBgjhvzC+hN7hoPJsmyE4RzwTFbIwgzEAzJ5s30ZC58X6WxTXiE0EdhlkZZqh5i0Ed/BEccfhsc4tawC58KHxUNB5kGcSml/iVVa9ApXTLdyAbT1GoDuemvHcs4S4t+EfQRdSTWObTr0NmiXfhMxIiJNnx4gI9BvPDh4Rn+PSpAID2M0stU7sKrt3rdgyMi6/0Vvpw/kTRt3aDpDJAWRruAq+QrN0qbJfcTMikmqY1kgPMDYIMlpjXCPzm/AuLhZwUKPo7Uc9QnuOiDNCpOOp2POhK3CF7H7ovhEr8Mz0d5jtBlIZJwdyL/fdjfe9AtbaIiwE4+X/i0wJXQ3kNJxxQORMwxtJCngtsPurS3+Sp8G1LuoRsy7s+wEFAiDwT5HqYaA/vM+RUeoS12m0vs1TwYDMxo/t3ZwqHSVThKZw9KJ2xaj17KmeFWJZ4Ogm8g7cm5UqcBNvnTb8NxIp/zFGVWAIts4UMs42GBhsU9VUyJQ+MtwAoqGD3tqgIfX714aRLIg3uKQh4cSBJ50JWk8eBYEcL9Q5VsB16bLA1iW0iQSirCP3eOx8fjaRc//gwtRNv6DrSMpWUltngwvjeeTnWLnarJrJrr1r3pA/fu2ID3WPY3Uf2JLu8edtdCe4PyV8thbH80HR+6D/Djh79/rNn/Fx32HGZDx9QeSGGLFgZQwGQOPm05WIE+a1r2N2UOHbIGYLjDMrI4CUcFLWDYIaf+QowvqaLPnCzJ+XAPJMajAS5MqRWe7oGpHh/dP3Jp+4heI3sAOjS9Ovyh7NbRcMsZe0Z7VJLzygFQmn6a0F6W8uNBmv0C5CAGXu4/IGHWWT0H3f0cHuJJhUuSlG2SbVKHb9QlKt5zWY73wPQfdPtnFbucFqIISXwehf4VHotkEQwsTQatFYz1uoWBKn/QCsAYxtNKcXgOrLbCozsbRjK1ZMs9OtcE7Q+c4zJ6e87d69b+I8P5JoowxpNEPNCsHXEXM5v6h9Ele5CfiMpPPh2UTz6tPZK6aqm1PjxslYfTunPAD9zDu61rdVAPZqjPXfWg0yT0wSFaD3iyDlzr0Qpp45p9Pn60IjokBbXKrvEE3ef7OMtH+aaTj/oWxYwM42rJklumWKsCGz1iXSOObBIimaU+vyaDLfT1ufY/KXLriKVBrkGsblprqM4dXjcdg4Q+VOpJQpjMcdRSsliWE0KcojFLYLEd4aKOaVBWI6J1tCc+p0+XvUGr22KXB0CtvwDqvoLvK2TDrtPDrwelr5oR1FyOj/WjPZcnczxCfjVoHbEjJHHqEU033GhvOG4psfHoh7/5W/aazhbjxkK5rgMtsTptSecuAXVSXP6y8Fho67q0sqoHKeqhh7/+jTzmIoshaJTsNBy6VQDM44QH3YODg3GdKcoLKkdVXfrqkcEGEtNUeB8A0wM85UcH/Fh+QPZh5YTqppOIpRPB1bO+MPGJ2BuL9AKPQ9KR34uYR338Q2d96XjtChftutZJA4a1zYE6GRw/Zd3QOTL5pOoKggjYrwJe7wTicbToQn79hVo204gxB7ecUZbcF+El0DPrMiBmtnoOJIvV3US8FCd0qusVui2FWF9zCPUeHnmu9iGPdCKKWBeQpHnpetVSNWxIsyAzwmC2XyOF9spqQAcaLMmw9nXlvPHOzuPXr2kXzdoZh+5VhznysC4tNsfqE2xVOhZ7r3e/57J/Jj1EDm71TsMb4JLsKXwcdwELRvuHrESG1zt35m7M5j22Kg6mHiAFM4WprmoSsVX5FLhucICoZNgs8VxRGe645x4cPWQ1vlPcpX/JmQlxyDQ34Zl1BItJ4X8KuDhcZ+yHeMhA9SK/ljuTsqOEQwVdIUT2WVqUO3gaxoSCRMbjPVor9KIedF0x6yjDuHPn3uTInU7bTQs1nU7NxzvaWMqPEd+rLHBdPlV+vtwDKe+GF31ksYMj+F2N4/gYRh57swRcIgcJmK0UYV8z/dznY/i7KndyUCay651pGKZIiZpJkFXLVIs0tSrxENss6FhDZQRWcL0C6aSpX6JlWnn6fqHFQrfGBPJV5FG2qp0CZxW6N1BPjIJLu6dkTs851NDcMC0m6AXUiqhLjxrNFXVYRH4u4/QYJBkz6fAQ9OB0+tAUSZW1RMJmup3izz2aBK3hzj9fCNfjeERbywd2H5euTUlADYJEc8zdyvIydmcBTQDRTQUnqk0VA0te3esVjHhUBlvMa38X8zLk6dlIZlp0ZHoaJu/g4fTdfYJMEmfDKIkNozDxaHhAXpPzK0R0moYLWRODvd+jbZX+Yfc671xXj0iwwGCMucGYj6E7penIegKF4FhGc6bKHvSP7y7nJZ4Db0hTyOERSR+AIyPgbEyHksuzqPJsmeQqk2ym+x72guwse1nVReUh+Lz3p5tEx5EpNZGRCtrbINp696VsQ7l5yDtKWLfL8hETRZJ87ogfSZDdUjPUiypdIfN8d5R5OsyzJvRxqnbhI3WqRCwoFh479DFxvuAJHoX0cKdiNM/EwILF92aYpBKILI25L58mPviem2OUiOvBsAD9LAxnvvgSntqyPjVGJqzc/EN7OOFBsochADy57yRmLidq/1GB1BE5rpbS9/Bu0+8jl8fnRqMCGIqZBmhSkyG0pgYanG5VBUe0IevtYitSegUs49cckGpSniQdS9VQpADW8yt+0iD075WxkBIqQ5LSOR9P6qW+aO7L2IaMsnQT0hsbrMc6EfMoESBAXaSvhgWYHEwPaZDVtptHehPoHONbwF+L2slcTM5BoyjsNoxeL/nmlmuXv/JaMyGsbVQZt+T3saAjIUaxb/TjTocUWBzNxhZFzg7vKwdT1g2X1Ti8dI6C4Qskl8QmehlYJ6eMTgFgzQbMHMLj8cjeA/oLo0uSAZjsjYLhbTKw7Pbg0coNJxnm5TpI1A6VgPrKS1IHBKndwllgLds2RoVdsTAKmekhvQovbOGLxchzBxYId6uSUgJNnr95+VWlxhQ0bD0yH4BGaD2qTqjJ86r4N+p1tIYAIDmM6hGYNC0djFidDluzKGsVaR+tr7978eTFCcN6oC1MCbl/dr22o3EMoPHgGgGSGb8Aa8IjD1YSSz6QD0YtGoo01dD1Dnc6VGExXUNMoxBNlSoOb3+4Agv0YGm79+8yT8RuWAKnx/KYDCA7mXCgpsMOKv6RDCMeHnSNISWU29mvYUfDecPHid3we6WNbSlLxVrTVmXIYTkDUoOYFA3MZRNzDaxvZHGCj9+PzdN2IbPJ1qIE7vYWx15LvSEtzEMftMHAevrW6aPxBGanWHKfnQe4QHIGiedjyJHHizC+ZR+4Lz6wKLsatzuFzDLssC+/+vbPqSbB2pIEm6uerNnjvGJ+JmZU0ufwicO+lBWQuModF1hsZuZ//LvEueU0qI7f4ABrelyO5Jej9loIuk4CrOILquig1vDLUgYcCqyrCCRZ5Pm3GQ5hi5KxBtarZ1+cWHmZvoPD7vpBaVo8mUzAAACs2JaRrc9sldTfyXP6AWz4lmMFEsz5kay6gXarfH5DU22ZNqDpqRxJGT8bpt072DDtfAXFtGlJTmmyKCqcn7Y/8uXWdwg89xN2KKuk1Dt7DguIBCfXuNLfrYCP9N0V0MWTOIzAj62UAW76bygrgHSoLgSViCC0oIEtf5ClIugXbLMNG6g38Z0CTt4BPqIvVfjW2RaVD7iPHsW6MWtJDJgIVLlS5BDrVjxX5IPbr7xZzl2VfNU1FCXx/wp9f5tqkWDSMmaNyYGfyJqiVBVLDra5StDGI+9a78kgkrWZNmA0L8FKImqwlJWAo1An1ThTldrVcWHzbGLnFgJeHVzcp8OKzF5kWC97HE7wbBVu1Vwx2aDNMibPMd4GuD31QzwwM/NijllnY/o2xsSttqMtmxsQoUpbyAV6rAxjW6a95GKtdlrollrIVKlPhKrRjOEOxEDkochW6ICHLpgKmc8/RY1qvNy9q+7NoPwM8R67Ozhs347gT3x5tktw/xbELoWTROdrH0yw2O46oH+PO6zn3JOFwAayGpNC70KkcZjcbmwvsbAizkoX+r6FrjNKtGxYdFUgnKZP6fBrCoLf0EsFETDrAwdQ0XXu56goMIEhjUBVT78NNh7DEPGlWyBBHiA0pNJwTREzSx0xBOZFo0NXgtlGKjeJ6OZO1Pxf0qDQzqIDkrfsQbKXHi4WzcxP4PIxuBPShKPf2cmLG5aPNsPSmthWl27lkht943z4p0ArdJfPraWDlgxKuvTlPVzy8CSVFDnuHYBw6B13WbfblbX6AYGCzqbioY1aVYltRQV1RMUSFwCS6uiAnL5r9PPQoHmsxupydWnYp3UoD4Ti+/LGAYfJG7367FjdStbRo/GpTvrmdYKBVWyboYVZfki5eG0Y/ov3gmGNCLlkmGKqlnGIy3c2PDxr33bB1Irr8692dNm+eaCqTFFF6ihPugJRnaj9JD2TE9RzuoACnEPEiKkKADF9tZafrGVwMeVCXlHNSwogYclacMsDmfsnfw4+fk9Xd9yAoFu6IHlBsk1C/F/iceBYHjq22tuUlsI02E0Q9frIgte3E9PfySuG7DAi39bvyOL8ROa38taM4l4biUlvsaimL0+PPpWeFEF9jdmbE5DPWEKpbwhYvHcglpk27IpudxQsDV1kYp6w0rVEOzeWMKswNF2hmJvKxTWKapLqOqOP/xhQmilBuIkXdcm3TStNq4WmZcjw6MmtVvqPv/31f2ffgdt1JI9oBqm8z0PeoSW/2r3u3gEmzd5m5ZvM9q/wDjGGx20ROt5lkaRYVyqRl9VgWQlVVeA2SsJ6CUY1jP8K1nFCwR6sz5wKNkMNR0e9s4THWA034dsa24CM0ThVdpesFJyH10xcoS/kvcc9osUWEGd0aQxAfIamSXxlyy1SCvsoXxlWsCMLlsszYeLiEzws7VWXnbSbx4cV9wflpdswq7X4aSyzbxkBz5kIcsT5sw1dkM7cfLrHvEKTzvbIA3PtYf/+2Vq4WEH/8kYCfipv3ExsrPZL2z/JYEiu+1nhcr0V0GatziiFZ196iYy/l6+HqfVcp7wTOgjlvSdxsgXB1QhNKwXkkCwI5Q07HKvN+6G6pkUfJbsFpdXJ9/igmXzXD3WOZznkYL/0fHmkezQJsyAFhwtvMYQVKGTov3pxahwA3gbpujjJJozfQqNT3cuSr3RIic0dVjbV8GQIuU29T7TYTuneW/uHf/tfel2sjkh3E94gY2Ql09Loepipf/eoNr6eHt/hQRF2otc39EEVUsvg9YW28t9yHwe6D7z29jZoUON5TN4DmB5odm/jd2+zfrKCNDT+OluMcQYHh0c5Bl5L9UOl3cSE7hAY3BC7zKtUbzRw8qo4m5hWFZotIxhvDe6wYyxYrNzw4xzql9CEHGpZTR73PTbAlyVoS9D3evcBLv3R8PFEh7bKPJn2whaoYXkaoxFof/j79k+zFHkN3KpfdITjuYeSpNel8R0ddXOvCLdz9Yu39u3L9PWqqKy5lTWmKvGWMHikLomGYamfLU31BVmpsAeTNdhuZ6F9FcKa2Wb9jFvZYeORrth3g4Clxxi8Bx1n0UaBVWi5L426Het72jp8XA1X3gj5R1nbN0KvqdtTOjKJxcwB+Te+LmuVmFbTTW/cVuNtYur66NGcz+Q9b5iyvc/k+astzIa4GNmauXzCTrlIJiKgeoM37pHfMzfFMZOuSWlj9ZosqdqpzZdE7e4qmxQrFwFe8LJBVatp+7i1LKnbUfu+RozpCq+hoRqnZgRPXbi+rR2FSQHKeLrXoGrKOQCYJgL+YLJFLkAvL/OLG39FYeD14mIxwtrFErGUUlK7auTkXfbxe8apRBQWhzKu/tjCEjsw69rKMeUR6k2jyiNnWw6qsWLwViNsuilFjTQvtLB+pJRa2TzQ9vqFfRaHWVRk+EijeV03Kn+zQvtNRR0fy5Ot8loYoxYZCEwsxXiLYJrFEcEOe2qUdNS101TxSlDFEd4GwxPnBpP8BrOAqj5MvWqklDRJXo+TJD5WocqFf16udOfTDIE3GGnPKwyhrJQpOxssYe+yLnjr5fOsjg4VDCyVaJpDP2iGblRxbErSeVz8nIfn2n1Yflikv8OQJtIA8OTPPyn9pGemn6yZvkzWzQ+3blrpnKrpnU28g8c935HOo2pHtqo8CQ493XUvz/UWRbetNeWxN/xnW+aqmLbHTWYqjj0vegSDlMG3Yt2nFtY3ZCsY/HVJz7bbtwOsXIYK5F9vDzkWKaaDVunyj7/9j/+avRLyylWwalAs+pqJPbdEpXmSqNXeIIEa7ISmSBG0qpmfio6fgFnAY7wm2AYLR0pqPND3WV7nmyWwQNLc3WqNG82o9XK3ZGLrK91yATfnJfTcaG6DexcHo4W7BVI2xNv1NWgUn+bgRLAPvy+Hxj/8YSvrf0txuxyhkpNjpjHYdeO5SP26v8ETXo4O73YbAD3TUenNYJo2zlXRA7wF7vRbsGLlqf+bdvcXerOcdHC14HaRBRqucwvgl03xN11motmgfhuOlXeIrnZp+SdZPHK9uPqzUfpiimdZ/KuR3NGVB/Er92XA/DtYrBKZT8dasF1ZFOsSNbo1biSmRleyA2cy58FM2GanHab/HTZfjgMi+Ky9BpCqpRn0WYAFPlV+dQHSuP+kfC0IShPSKChaOmTgdpQ12pH2X0eRakcSWqeQLB3lD1CNqXWCQuG+o6RXJ7czjNIuIhjJEC/VifnT3B5nhtF+0kvkqhGVn/ouuXIKkrpNTt8bZ67sWZmggTK8wBbL6oVp6oi2WOJ5mcAu7u3rGAvRydcB2KogCEkeetURaTdEgUC3AAGAMJ9cGbsP5lOq+IKRJbxhBmxJjHzs7tLFlbKfNfrWSbIJ1layzYtO1HCHinDN4Z51mgezWb/dfiTEVWocBYet6bwK3xAUcvlAGspbMhWH6/vNir/tRjlGOlG4WGNuAYIojbEMK6CiWhRzfVnkfBzQr4f5Fhdz9E91HVtKK8G8knIXtYqO6nVZiSLIxJKP1D5jY3xhqxsGDfAWcQNdNKp6qgNdUHnkNRlKxhiHCtgZSdD8cip5YKUpJanUU4rqpTTbgjMxNTcuj90V1eZqSFYdjYplywsJ1FUxLtHnBCrrY9VoWKzr3V28FKZKUbmdquiqRC/AOrm8VnKmmEWnkEFbcD4DLrwBQsewaM26MsoMUAM07QhN+PCsvWPuniK4xNTvw7MKX+DNAtLQTmT1VERn4mpBOrqh3nK+K9p83NEgSSwrW7sBx/jduNlytPFqy2KX48dfcVmcLphtuJRyJG/gS6rFN40LMmv3Yx6pjILyfZiSbBN9m+ScR6KC/iO1lltG6tbdGLfNdXGN3GQPbbRppnrfia2SER4GQ284QUQsUKjAHLC0VDLKn4/yxzvNuXINiRBMH1ypR0arZmgEJK8xRRd6Yabbupq8JbFJjavVaatmqZt7urJs9VNfzDDbDO9DVnkVbJXDoFrd4BrlGHKv8T6y+iR2dirJG4p3DaYrLAolFqQEkISHOFZJGp2Cnzs6MWKtqGlSqmXBABAdWbLVNlFb6mVYMwR1v4axLIMkeNcHhUkogFBehXFJU2Nt1UmHvRvQW4PwvG9ebqJATdqmxC3CT2smPCzkZSd3e0sX/V0ayNcXN5hd5MZ3p2zxN9j0OfxPQ/5sVLddbKPKeKeoJtvWYn1I6z+X+mDHSKjQS1gUqtavFCZW8cLY6Jqq6wPksZ4ebZV19L7WGf1E+KTNpDVWolG41YCdl6bVg6E9nHaJn8HOH1Fl5VsbXlrrN6Ks6UIBo99FKMs5694bEFwjIHoLpFYGBCqmPPPxuO9mOx3jleSJ60NGaswIaufHX5xFt1jHO6PXT1999/TVa1nDWtVpnYKPN0Lc2finKDn52BMx3rOnwkqYkz8RPkm44H2osuqu8F4sysYBiUw53743jms33Cfh5FwUtwSfy2LwsoIIxjEvxNg0YlEI67HSZcHnDXXdXbQN0ASo2we1mwwIKNBamFTNjPUFSXOVrUtwp9VRWWkWBAJPaxSDw8LGzQNLHXW7farHgKiISAlS5AExBWIc8fUZO+q2K6eoJQ4d+Y/cREu2mHbijD2MzFpdh/6HoYwG90vfX9+EnG9ek71Uh126CEBDgPEr0vJ5FkzmI7nMNL3Bvft39cRoooMa/eUcBL8VFTIKvpJAbSqdFdO1rwNjauox9SXxKXtXhXWxFJxYl4qIqV6gNUby4tYRHqLPXwsvRnTJl3qgjwT2mjKCKHFMuCMquzUYVgt3m6l+SmBoqhpqlqAbqZH166KAeBYoOXOnPkwmF4fiEv0ga19VGto3mlgbyuvDa+aN2OVCyyCiwD5HBt17R3+/wfzHyw2mpjVP0yjp7+/PgGQz0OPhwhiJOaj9GBQRT0Syj3VJknQfQ80oYffX7+pZxvt7gKjsco8v3LtHKEGpyELlLgSY62QOMh9n2WHd8N7xcUm6wuMyqYIcspHfQTKDvoLJZgme+eZYUlXTb698b4xuXq7Zj09qt7hnMZh7U2E/aP/fZACT5hnP0vnAxll18nE27D5hs9ECyAAvsW6qILE9r/wo9pCKkfZEZJz64HgxtqpMg9qjxDGAaPiK4rde8FjySZmDzJt3JuV646d0j8OQCKiQ99beXhDuAZZCeZeBfATdbrPLMyUmAR7pHdyjdev1V7hU19skgsl64ANzhC9On6Kt6QLSzeev3zwBpBp1xYtpItkZF4EWemYOmGblK0JT9jm7263cnSNLdWJPqkC5g2Tg17143JYBcyARuB9mx7l4GPK99929B3tnn/3SgeUpVuOXKDIwbwi7MC4ANBT/oq6I5IIvnBmlITRszYBNwoubxNddza2t6yGSDqln8lGNeZ5tuLm7TJSKVs4Unrapoi0vHrce58io1MpG4ZtbGTjlWnBEQfg2oYACpkMHWLsCbLMx2NmUHfCMKqLJymOxrGUQOo4Z4SApU7WVmgVSTSiRrJGCoFnUrJccjaR/e11akAMNjEaEkrdhiWpyQ6kFaC0lfzHwnf8D0NHxfg==", "memoria.py": "eNqtWM1uG8kRvvMpCr0QPOTSpLzCXhhwA8de7waxsUDsHAKGIJozTXKkmenR9AwtWuAjJECuuQQ5BjnmECA5BLBeKI+Qr6rnj5Kw3gQWIKmnp7u66quqr6pHKfXGpLaINUUmobywBxOWdkbbSheRpsTSdWUIryqHmdiSzovKrDVZCm1RxFtDB15VOT3IdaEps1SY3JRxQaYobGHcZDAY0QsNaXioklJHlvLKRAbPYbzGwv/8+Y+/n+LPH8hkMyrj3EKbEopg6Zg2tkj1mEpzU1Y44DAgMg5aJaa0U53FqQ5jm1EQ4k9oU5OVoqjNeVonwwmOf62d6GtCnjQO8v05QW4Kh2VeUDIN9dp80FO7voT0ITlTI8GiM0q0w+mpieJIi4xEE3YlekK/SEwWmYKXUGKAyphcTElitpqqjFILfK0IqbdDEAADAhiPGT9GWXS6h7hJ4i3+BXusBbopm2KKUGe2Nc04QBW7Mg7xEPD+1Ja2mOaF0ZeVKw05nWDSXNpCjEorbClglcn4qFxnJpkM3jbW8vR31m6x52UR7w2bwmdAbMZO+RlPZPCNYUwo1AU8ziHE57BBk4FSahCnuS1KugTAzdi6ZlTuoFwUZ9t2Ik7NYLApbEoTqueA1ybeDgar1z+8+BXNu02T1za8CoaDV98+f/cWLwKls3Bn1ZhUooutDNbaGf6fx6bItJOhLv1gXegPVkYhIimxajgYDCKzARTlLhjO4B2iSGyfk5pCjxKBNZWZ6ZuDoKJkUbyBURPeNoldFBeBrKkl8E9hELdZu+jSxplfg8N1fhGtkFKInfiDvjQTxgq68D5sSPUVgqVwgcdh8sNv3q1e/vLXwP0G3l7Zq/k7ZKNf/tgx97epR47yZidWR43ZZXHotH8flzvkkskCj8yQEGKb7n3vaBY4EUEbr5K5CU1e0rfyD3n3AJNbtUecOjWjxRLacfT7h2OtltN7E6S1Xjlc4ZXwaqYyQV+SmmDsvdGpiymIfK/uKyxaRlWaBym4ZUwx0jYr5884mF1VmJV2YRzPX+nEmdYRoLREh8YLzRvQNkbDDAP3IOtrJWvLFhsAUcqLydaUwdWYzoEduIyucCRJ3C5rOQDBBBGrskFOwUXbCmk2JotdHaPNlarPQG595/MUzMII9hgUOYjJlmknnIe8RwMrli/aKM8KHP+3x2EHnKRZB1WKPT4wuqlF7bLlROeAOQpuT0JBlWommTwBv2x4EKiz3z49S5+eRXT2/ezsjRrCLZ4OsbQJV87VTGN1NByfCmzgwOIOGWWv8Ly2NgnsFQvsYMJ897CYXZyfL+9JFG7EMu2x8I+QsauyQ6UzURBHOVOucu0kIuulwCx36r6GoD/dW7RBoDgRwvGM+aDDneMh3oCmuQwhNBEOcIB/J6tPRX+en+541Gs17Kl/7BxbZ1odkaXR4W5VHnIT+KBk5Va+gqIutmH4ou4BmtLlo897d8aFDDEKb6CIXZo0RwGUHiGxDgVkW8XwEvqDJkQZnJMovYdWQ7eZLWXtAzqRnP0fo9lzThfNagOPnSZ258kTFH7UVShHnF8zahnAyxBhn06RRxzTs5NJvyHIUpdulbbcfc/IPR5Tf7xP2zHI9QTIPeksklHaqenUw/oVqOd3/8y4v9vpg7COm9BL4/Lq7q/SC4Xc4tV9DrDLLYiVRqOP/36LnuDu77SGu38+GpF6gJs6UK/5pLi4+wvVlQq/4DCvcRJz5zYHt6rR6Bb9VrAfHr0m9PEfJDPeiCdixBMxFkse7fzqA9CsjEZqKQcwO18zOzfJSEqaTx7U7ScP2+6zj9K1aHYjMm5Yxp4Rvll0/LWk+Zyul+0OvL52p4XUXkGIq9IA+8Bwy07atRuerBQsmqDdqKd0ez0JdR6XwPqDCYbHGd3aq+NUQMHmo8BPwe2z83Ma8UHTKbXvzoY1xF9Q2zz2WkZu8wSIFiW/QpBqWJMxKhj27f8BS42zxOJNn5aH/FaGJ8DVmtM3c/rqs0H4inWQFv1WTmQMH4erAbOdkSBsQMSNwD5qNmdZo1bP0l7tGi4XT7+eLZsEFUmdfScaq99lo9HdnxIQCE5D8qwTw3cDdiDuBuhzSzNDaA9Pt9OXkkAImJvFk8YJT5ZsKyY6VTClOvVFkWWfg3C88k2miG0KB/LMpinrh35GUJSb26pAbEYo3LjX4b5kugry3HcivjAUJpY+WyP6WDOpFILtDDv3yDr040Go03VsHXfBeVJXh2FbRX46A6ZQ/NHobAJQehSJwIdtCpduNGTL1uOQdHuNJiA7BH0/dxDyaY9SwmlY9kjoE+Rz9FWzQeP97sA6HNnAfgAtajlLVlmIHpp0QVXvX6isMnu9ciaNcaFl1XydaS8Cu0MbehZ3R6pX1iVde4bohdsXNUsIowiNCKMgccqq9ioF0AXxSxf1c1NGKlxuuwKkSyOWDfoYYddP4J/PyEHLE6GnJHQxe1DUWOsFdv0IHVHLKe1uHZaVTvjO2RjDJ6FRCESXIXoGXZSO+5tAvSukpUDxxJW+BaKVBS9xVLYvWFIjv3+A39489mlWJEBX8QDjIkbx7JK+kQfByQvlGw4uY+eTZ1+fotFGmIeTEWERJ2t60bXx61C/sOhIwcnXC9AUKCLo6TEicHRLysOuTfSx36SOZ9yOih5JgO6l6Nh1k/fDv/kUBarLE/7wETiA9PFv6Ft498d/PdCiy1qvR02Ds16ycN2JTGTd1F4iEzb8WUk+9pjMf2PinpnPczNhe3AkmiYQUhzah7bUB3zCkFatWjQFX13QrjKuK2Y13bcsw3nmeR+C5AJzwgZq8F/zOi03", "estudio.html": "eNq1XFtv20iWfvevYJTuFplQ1M1ybMl0JnGcy8CZZOL09ACGkZTIksSYFzVJybIVAfML9mWB3X1bzMMusA8DLLAP87YPk38yv2B/wn6nqniTZHe6e7eBbpFVp06dOvdziu7De27kpNdTrk3SwD/aOaQfzWfh2K7xpHZ0OOHMPToMeMo0Z8LihKd2bZaOGvs1AIvhkAXcrs09fjWN4rSmOVGY8hBgV56bTmyXzz2HN8SL6YVe6jG/kTjM53abcKRe6vOjkySduV6kdZ8dNuXIzmGSXtNvP46idNloDMf9++1Ou9veHzQaUxZyH+/DttvZxbvvhbx/v8M6bncPr+ki7d/n+/yAc7wFM7wdsAOXMbwxx+nf7w17w1ELb9ElsIwO3L3haufBchgtGol344Xj/jCKXR43MLIinpjDyL1eBiwee2G/NZhwbzxJ++1W69vBkDmX4ziahW5/zmKdKDUGTuRHsXoHNcZgBLb027vThZZcJykPGjPPbLDp1OcNOWCe8XHEte9fmQkLk0bCY280iOY8HvnRVX/iuS4PVzv302i6nEYJ+BiFfTZMIn+W8gFGQcx0MfD5KJVPsaIQj66XTH123R/5fDEYs2l/H4PM98Zhw8POSd+BwHg8uGl4ocsX/d6AIBtXMUDpP6sdK+HjZQXNxqmFSIyB5Fu/TQeNfM/V5CwJKJtsxMz1Zomkbcpcl/jdnUrSOtOF3E4bztI0Ck1rmIbL0m5pDPZMWQySK1yGlPPdW2s7PSpthGetjV2kRLxwAkYD0yxOgGoaeYITVxMwpoFtHN4Po4IHiigrqpB0v+N0Rl22IfTKmj7Yx4Y+d5cR0Hrpdd/q9rJ9XT5iMz+lBdMl8bffxrPrFqKOuc9Sb87F8NE6U75eBFtodF3NCng4ywUcRiEfbCqZ1KiWVLb2Ns3/lTqwh+fAC6Wz6LcPaC5gi4aytr3WfFJYBJulkSDeiqY8XDvB0I+cS8zSoMZM+SvlUIUZqL3IkFO+SBvCLPpkRps2LABc7kQxE5yRfFLE75NeEcXVA9KZSnwSS3ItvUMFc+L7Ezpy5QhyaIsGZouSgPn+csM6wDeW9oUY4Un4YnqLHkWXuZjKOGiYKG5cZfJoAc0wAknBFp8kJ/rC1oRb6hGPyXpHURxIO4ZO8z/qDcwYJe9TcTTCFblezB2BHgTNgnCb98ocG2mMlOrB3vwKFLLQC5LlhhMkyVT93ODTLEm90XVDxTCFOUORKdCvtruSq8rUpLNmBz/potaIsrbRhVC3VZBiXA7cH41GQOWFo2iLCGU0QdASNr9HNG4qFWkEgiaXks6kuDugc2em27Z62CVJ2ZgPmTvmS8LXI2kpALLmkuQetYTkfD7mobu8zRWJDZWaPVpzFru9dWdRNkOVOTjOL/ZVubmvnb/iQ3Nm5Gex4N8015tXfG1x0g0Zy/HKiooCb7EDlaSAGK21yhB4y8zRFTzf4q26pRN4oZCgcqX34Tu2ScILkRPClVWo20LVdtPaVKeMZd3VjsPCOUvWffpvAu56TNMLZdnbxUGM5Y4mkqOMM6Rh+5nr2c8Ton3KL7SNBKNsegdVqXbFCmkihFTqktS67RuUI4k6Zwlfu8CnDeM1RdAyf6o26LYkdBpFfrKsurZcvRsLFQ21qqPKDtXboqvqVGsmWT5bu9eqHKqq6LRYKW0FvnrQskV29mCRxQl2xQmyVOigs9pZ7Rw2VdJ/mDixN001Kk3smhdQaRGwae1oqV6SWn9ZSycx57V+bZKm06TfbDpuaH1KXO5789gKedoMp0FTAP2mZbX3WlarOZx5vivHrCByZz7HipopUTXBryhMmj8PJV+wAGl80vyUBM3aaoVDCOJxiqYsn6h2wBuMWPNcuwZGo/LRNDHg+CxJ7Br0sSYmqZBKxDQApBw1l6WsMcdCvqhlC6KwdvQeajaL2WFTwh2twyNLBqOO3nr8hiU51FbUMqQTME+i5FaM2Py6dvQkdjzfv33fiQcYP70d4McZc7HXMX7iqELZYRNM2WAO+LV1wnUl03wivpZvo2Zh17Wjv//Ln7TT2Y3293/674KcEgrKl7ax26cCWBSltaw63X4cAlzAxL0oBqR60nS4R+MOlmMVfuB6HY5lT7LHW7dwYxYg6XRY7egZHr/8mZ5vhYZDCQH5ln40PUYJnrBYS6W6FGRJnt7JWYcFwXbG/s+//sO/acdf/oxww342dx271jJbtaPnVMbx7ecATHtfQJ2gEPNddivYgYA6ZW6keTc/WnewHcCNEjQin3Ur1m7PbO/XjrrN3VtBWuZ+i8wh9oa3UzdCEQldZ2VD/WnGIzvfzveTBfk/iPNOnmc4pAAq+2U/mTuiwC7oG7MQPIFX5H709z/9+zpcESpqmnDTIPZrCsZKWtirZGGqDVOkopsp8JY0uRrFBqJpM4E4r/otraW1KTNTmPf3c8dRHFcErc1xisebozK2Fe6axkSMvcUj5X5cBOw11afxOEorPvwJIm4Dg2zT9oX9pXAPxzCSuASwgfQK5RHMPfK//GUchVv8t5DfJffJln6ccZ+n0XaYCciDaf/jX7VjNq0El8I5Zz+VGC2DKQ4s47P2QGOJ9v7lu5MTbRRHgVYXQbM+yOaXb+Khlx5DlDEYtSoDZWHYUZPNCijCcQnLi9P3z0+hwEiWt6HwxVTSLMCqy5+9e3L85ivWl+CqCN5FUXASzr04CmFq6VYkvJhPmmsLJLYdHDVJtW/sxD5yI2cmZiCn+PoMsnLSKNYTY6Cgnr86PTmzl/Dm/fO6tNYPV3xojf1h3cwGSEdhEtVBGQC8m2ziwhQZAtBMRYqgBlUuQMOUDKhRCv0YYjL2q8GJRzQwDHxgCPhqVIT3gjjxKqdW2SF+9+r4xF6uk9+vvzh9qiGa8zjSMKjpCHZOpD3Ufvv25IWx/XhykRrRdCwztReRG6Wm9l7keZ+S0kr4JGs0XGDV86d/pGa1xjOL0PTvQySjpvZ9GHPmm9prD6ldhMU7mlpuDX04Dyx+Sr8ccV6OGLczWZKXeKWNCmCkGBTVIysafgLkm6e/BVQqontU2vYD1C3mCZxr17WS1Afo2ftTTY1++a9Q6z5DRRSUyPAjt/2h11IknL55lrG11/q2AtX50O6VoILZdQbZ7n0raCgpR78uk0kN2q9NZlAP4Cq0pF9/zaYMYVUTY5grqwuIhqPXqPvDiISq2tBaDGg0oOFYCCGOOJpOvIu5i9e4dMBCq7KVHIGSMkpwPsEZgqknkh+k29h2/OUvDHqkciBtEsUhcnNGCOeey6MPyHS8kMVWMN0Fxj/QmFQqQUUONobldpH3l8Hw/rf/FBAQ54cp5VvMmoZjALwKEDRD4oiY1ORkfZXbfCz0iMd2yK+kv7R+4MMXp+/UuL5kId2ZeCzpp/GMmyRyHs85LOMKkfLpbDTisZhawUVk6FDgpm+9BfffUatSf83SCbUedHkdU8yYHcMYlBedIa7qXhjy+Ad1ZYPHl8IjlNFHs3Q6g09G4D6jLrktST979+JpMVggRgDhUI0pCFaQT45Pzp57fuA574u5EiUioGPY4qHomtt0wsFO7h2pprKwCguOJ6jp9HypCwfrcwLKXWaCKpyXOHxG77oxEONWkZGUQMQp9NZCXjnlmKZBzIMS2NvX705evwCWmJGXzoig9hqBw/WjSLPFKotCg9yZ1q8FAt0wUUzuGpbUUJ5tCLUsb4dgNKVO6JwfswCb6shSC2k1S9IidG2z02rltDupL1BVwqmODcytvAO4Yv4zFLgkOSECGqY2wTvkLCnfNnY25ZBX2+pkG49jr8zaF3h9yf0plLvdMneRji86vNuWD53ObsfJJIMAqtPi/AgjP4rKpvKaJxO9xB4UPPwFjwKextd6p4XjG2ZJ7EqpqEZjvl6+iFkZoj0exRblYpTGWgu7Iczm7atmJ5tDjQbWSzzy6AWhAiSndMIDr0ToS7wm0wmYfErCgWKNxD84chf/7O6a1l751LS8UN9ZWEL1LOuFM38NVwcYZqHlsCStkIgxaU9WwKZk4GToYM/uvkn/Ke8L2Hzb2Avu3nZ/nzFs2y4jwCIg2EGYgzOGqGa+bwZwN+qRCRSJvVwhoYicy7LF0TvMAPl8afRptOhi8CX0ad+kFFZqjsRGOTuhYnPmSd0+w8HyA8jsTYwXCaCe2aZLqYWYLGV3mBXjxKFn3EGoid9CCfT6L20I+d4waQqUzbGfjpp1YyDJEjuUNhYwxLrRLBRM0kRXQ7+kzqYmST61l6op0T8vpG6ed8xds3thFhKBNpltq2WS/C/MrD+hFnVc1+xaWNYzoQC0bDhy97HM6pnWAS2Vy6jMyJsUci1r7cGoAHHe2MXGYvFe61GL9gRGGJL4V67PexYlYrvY47xr9sxGm9aORr3eo0cmydYC0e2eOWJ+grWig1FaZ+2a5y0s68lV0nSwbk/sKxetzi8vBiBa2AC5b2LxS77QT89bF9I0qK0eoky9tk/P2xdiKCtbhU1YlnV63rkwrGDmp97Uvz5zmM9i/SWCpBesY+1eyOEy1l1BAo1WEDc6D16a9G+j+wDIyLwry3oXyoZKxQKGH108liGkT/q+CfKqhGJP7OyN9Hu0zlhu2eOhtUdt1lzDhrA13aBbB2s2deEX4fBib/FDFCOgkhDpCmQBr3HJT4Jpep3bznDDQmlvwoNCdY4gxfXIPlqCmMjyEvLSRvZydklBypVjsLNLkRKHigAgisDmAMkFf0ohmVIcInIwxPD0GiiGxbBBGYB/LanexVxQ0G+sRJnMoRnYZqyCwk/gzsG+ehPizywEM/WhsVpRLoXoDc5GixKfR7AErrMbG7bBfbvdETYtua846tiEaczTY3FhUApqfxAFIFhsmEkGJLK0bSCDl7bM9dhCT6xrEzmAFI10eeXJhUkAiXVjPGhbB00xk7KQEgJrFM0fZMEP2a2BCJgjYTa7ySepQ8dt7pcHBChbMwDHWjwUMMjmdWZIeCdKdG48cE3Hui5m1cjNwxxmHV4eCUkHvBO4IaVHd5rYFWl9bLtNJD7idURvD3ryRerY2zj6JEOZlKkuc51cAcsWQvVZAhOheJbaLXNutwZ3qfl3392LrBkS9GfwnBb1ZWBeMgmyC/UazB/aY4ul2H0IZUwKVjnQu3SQ0rRomD1Wv3Ki2e3fsazZJQXM5fQVygLYb/S6uIqqG5ZMHt+/PrU/frMU/BZJsZ4iKY1OI/pm6wxbh2PdWGl4+PLncDzzUWv97a/aN8v5FqD5l/+IEQB4cjiMj75ZQucA9BzZgKt3MP3lnzUavN42eFMZDD4+JGvWiVJL1J3J449y39LQSpakycd+vQ7zZ8l16Gi5KPkCFViij4wlJKCEEtvsinnIK3nqTPSRuYR4JpHbr788efKsXpgzCp7LlcMICAJVgyLqrFY76xslk+hKp3ucUuR2WOgmtmjdnNPUxYA0auT5XGQyAxTFukpxtWgkwQ3olchsrAkjuo2lgB8NhjFnlyvl7GnMkBRl8qQEA/IUTVpL9WjtOt3M1gv1GKujq2yEfp7QOXRCCJsWas0hhZT5Rgkr1SfH6hu/j5tdY62iOu1W6wEXuLnbzJCtvv0o9RQbwJaMpQxrKJGiORcjK8wUud4aQGlisJYQrmQQgvGIFYO7A1K0kSlH2zL8CIXbDFlXcIwdUNgIsQ9EcAE6EqZt23Vqj9U/f85eJ17dEJFClBprFcsZ3KyLvKqoRGQHHInYwdBx9kwwbzwJeZL0rV7PpM8sffnW6knOVbZW/bo6neirdwRPIElR5arGQmnTRyvEMrlTkdmTYAbrYSsIhZcJvNC6FuzOXdJ1ww5CVTcVg3CfVNBVR4C0KBEGRYkwILUT3faKd6rXlfLAF1KDRoBbPg/HKaSqMOXHf5KBvKaJ/BSZGYjrBfu8/o6DJOpmvUTFy2I8vPZi8fskgap79PQ7PhYjZ1FMXRhqYj33qXWPhxcK+h0fqpFj2VCipyhGcU0LmT9zmXqSUE9h33I7FoqO2dso4R/e1y8kkZUjJlGc6jozh4Z9pAvCZXB4Az9h0Xe4xsODA+Pbg4PG2uywMqvOX7gch1xOeSdjqYRw7oiVF7bgquX43vSJmKFoK7BkfBzaeWvGgXtKueoj6HV5f1CnLKvsOyRmGBy8k8P1+gecXRNQUYh9UBrqOCa5Ll2CGhV1KLd+hirXE8AZ5ZlELx5nT/3yEVESKKzyaCr6U9NEYCP3DLd7BgejLP42Z1JOaLMVqh1HBJOLAr3Z15/2PQVTnhT3QacIT5mPqwuGkfnx9AckEToBUzZRAaYLEuaFiQTPw7m84doIAOvuonBW4hag/rguvrCp9+t0nVcf4HhFZ+lwb7dlVJAXZJBzqEPX65sUFEa7vvvj+uEw/9jgiPKDY+YyTThCzdZmoYzlFs08i6DHYIUW8BtsG5qazzTRPqZBN4LBWvW+UIH1I9EmsmON+ITYwONppG5Oj+oPBdSpIFY3+tKvyFwdVbqoviH0U9n7VJng4PYAK9kGFNuvcJ74PpaK7zu0zCaQaMcnDFnF0D4alliaRuOxL9XAHFp0kwxFsOY4mcgrqjlq5RQqsUk4lz2j12xKzvoXpayB/SSO2TXgxK9ehBfjcfEMS+oXb8o1B999Fwj7AnaiRaQwcsQwxAAVBXLArN+vP8yK67EorrMU0lBRSGVc56jPafEFNbJ0/Tw0nQs4iY+H4nrby66lS1fM3yydFV3Vekg/w5W8ufxoWJ8i1BkiSVSNKooSMvyU+EruJMyYgbJHeZbwgmzjHsuyLpnGEIbvvlMP92ybGeoZJYjL38xS3eoaA2ZR+z7VDTH6KqRBS2wkhCTIYD+lQ+WPvL5Wh8qu17bDDb8LVpeqHuV1olCU7fcoaKrDSmZcJfb5xS1aVVWlK1Qps2SiRwYkiZeM2Cv76MqSn/FnTu/KuLWLcC9T1s+f1zcoUl+heBstBv1rdLh/XjxfFBwN7KNA7CKcgg12qIJglW+X68fVelu6KPfM6sxTlnjOZu63P9wfjkZmvp3Mykp/7iAHSk1r0+XTdPJD7KW8L4sRI4/KV1U2yVT2ypKd/jci6ekgtSXffUVWtsp7tsLf2XXVaKyX7lw2VTH7W4R1LcxCOJURHH4ymqLonrKxiL1FB9YeKhVQyYIaj2x3e5RDyJdB5m6SStQs7KPFtvAqMBnCjiPDXQ9mcppcT74Rxk/meCAguu7R6+KEdZOylP8jcow7eX0uv+66uI3XRMgyl18RNfz1ULb6mn2cn9inegskMyQKjXGU3pnQeJSNZqQ5FKrp+yjULj+7G6ZAU3tz0rEWpmxcLlCHvHxgtR5RQylL/Nd7UynAU+sa/948BHCvJxtC5eZSutYjKtyApOKcmciSSyezkBR4yH5NcIOC1e9mwZAqVtUKNPlPieHOTKEsC9FuKCUJpEaVJKwMfEsKl8UKlcUVgrxL6Peq71vFXw5CVXBjJfdx0nhtnywD2yklvVU6pJcI7XtfkRYPtsKU6SK3Xop4ii6Vmt+x8UbyvmXjDZj1jXfyMFLuZRRXlqo9ru5LRfVaAOYxyxKR4D18tjLFbTD0Ob/8o8xO+cav1E6R5nFn0+RXhtNlYJUimoxKgaVimi0uYkUwUo37r238DP6/ic4obA+qB1Bhl2jO9GayaTj5/bp80MWxTPihzImxW6tnVqekcYKcwN5ySw91ohD//btTve7RpyZN+uyEVrjRVUhlil135Id84oMUTAiydNLyzZBGXxPdcBnTluQmmfjawN7+hcHdLfWv/ahk/auVvF9zGkXI8Quzc1NbXBlTWHjG/ZTpqnZHqVEu4d1NX30b92nvHV20b8VGUDvqDYtrZdkd1cv94XrsjT+IfvOnRFTc4he8zDvDRUBAdZpfSL8ZEnOsOfNR0soeNoudiTePks+flyppW+sAb/l6buOTKfPXff5nlr/sKn/GVf1s68LArvILh22EqsNd8utEp6/6jDVwyUAoSOAl8DgIbVTF4VcWcap3Li1WwuaNekN2vyUuxaZfGjGL/otswpcC5oWVIE/XQUOl2166Sglssmz1OXX967iw0bz/GZY+KhswvRR9FEJ+Prp4+FE7FH/libp2lOUaFg49hdEY9DcoYvIj3FW5U8aEa12jXZBJ9N5TyH8dreVKc7R1f4n998I+4LrOOFnDW4awn+iwcKYSMxoVvKbCZG7/ngxfr8/rxufP9ZTuMohkdaVymwzndlkyAhiK4YWufmkfybWXt63FNqKrr2K02l+mEpS4tuvG1txClhAq31XfgaUZ+TxlU+IeXa6kGaOHQr+KD+w3mqU8xcBau0l261a5iclrJ2oNGZQ85X+GdNgUf3902JT/i4f/BaU93Xk="}""")
for _n, _d in _FILES.items():
    open(f"{APP}/ap3d/{_n}", "wb").write(zlib.decompress(base64.b64decode(_d)))
if APP not in sys.path:
    sys.path.insert(0, APP)
# Si ya habia una version cargada, se cambia por la nueva SIN perder los
# modelos que ya estan en la GPU (antes Colab seguia usando el codigo viejo).
_old = sys.modules.get("ap3d.engine")
_eng = getattr(_old, "_ENGINE", None)
_app = sys.modules.get("ap3d.app")
if _app is not None:
    for _k in ("inline", "web"):
        try:
            _app._SERVERS.pop(_k).close()
        except Exception:
            pass
    try:
        _app._SERVERS.pop("tunnel").terminate()
    except Exception:
        pass
for _m in [m for m in sys.modules if m == "ap3d" or m.startswith("ap3d.")]:
    del sys.modules[_m]
if _eng is not None:
    from ap3d import engine as _e
    _eng.__class__ = _e.Engine
    _eng._ensure()
    _e._ENGINE = _eng
    print("Código actualizado; los modelos siguen cargados en la GPU.")
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip() or "SIN GPU: cambia el entorno a GPU")
import psutil
print(f"RAM del sistema: {psutil.virtual_memory().total/2**30:.0f} GB · CPU: {os.cpu_count()}")
print("Proyecto listo en", APP)

## 2. Instalar dependencias, TripoSG y Blender 4.5 LTS

In [ ]:
# @title 2. Instalar dependencias, Hunyuan3D-2, TripoSG y Blender 4.5 LTS
USAR_CACHE_EN_DRIVE = False  # @param {type:"boolean"}
INSTALAR_TRIPOSG = True  # @param {type:"boolean"}
# @markdown TripoSG es el modelo abierto del equipo de **Tripo AI**: formas con más detalle y bordes limpios. Se suma como segundo motor; si no instala, sigue Hunyuan3D.
# @markdown Si lo activas, las piezas compiladas se guardan en Drive y la próxima sesión se instalan en segundos.
import os, subprocess, sys, glob, shutil, time
t0 = time.time()

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        if check:
            raise RuntimeError("Fallo: " + cmd)
    return r.stdout

# --- sistema (Blender sin ventana necesita estas librerias)
sh("apt-get -qq update && apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 "
   "libxxf86vm1 libxfixes3 libgl1 libegl1 libglu1-mesa libopengl0 > /dev/null", check=False)

# --- Python
sh(f"{sys.executable} -m pip install -q 'gradio>=5.9,<6' 'diffusers==0.32.2' 'transformers==4.49.0' 'accelerate>=1.2,<1.6' "
   "einops omegaconf trimesh pymeshlab pygltflib xatlas rembg onnxruntime-gpu "
   "opencv-python-headless ninja pybind11 psutil hf_transfer 'huggingface_hub>=0.25' sentencepiece protobuf")
print(f"Paquetes listos ({time.time()-t0:.0f}s)")

# --- Hunyuan3D-2
if not os.path.exists("/content/Hunyuan3D-2"):
    sh("git clone -q --depth 1 https://github.com/Tencent/Hunyuan3D-2 /content/Hunyuan3D-2")
sh(f"cd /content/Hunyuan3D-2 && {sys.executable} -m pip install -q -e . --no-deps")

import torch
cap = "%d.%d" % torch.cuda.get_device_capability(0) if torch.cuda.is_available() else "8.0"
os.environ["TORCH_CUDA_ARCH_LIST"] = cap            # compila solo para tu GPU: mas rapido
os.environ["MAX_JOBS"] = str(os.cpu_count())
tag = f"py{sys.version_info.major}{sys.version_info.minor}_torch{torch.__version__.split('+')[0]}_sm{cap}"
cache = None
if USAR_CACHE_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    cache = f"/content/drive/MyDrive/ap3d_cache/{tag}"
    os.makedirs(cache, exist_ok=True)

def have(mod):
    return subprocess.run([sys.executable, "-c", f"import {mod}"], capture_output=True).returncode == 0

if not have("custom_rasterizer"):
    whl = glob.glob(f"{cache}/custom_rasterizer*.whl") if cache else []
    if whl:
        sh(f"{sys.executable} -m pip install -q {whl[0]}")
        print("custom_rasterizer desde la cache de Drive")
    else:
        print("Compilando custom_rasterizer (3-5 min)...")
        d = "/content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer"
        sh(f"cd {d} && {sys.executable} -m pip wheel -q . --no-build-isolation --no-deps -w /content/whl")
        w = glob.glob("/content/whl/custom_rasterizer*.whl")[0]
        sh(f"{sys.executable} -m pip install -q {w}")
        if cache:
            shutil.copy(w, cache)

d = "/content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer"
if not glob.glob(f"{d}/mesh_processor*.so"):
    so = glob.glob(f"{cache}/mesh_processor*.so") if cache else []
    if so:
        shutil.copy(so[0], d)
    else:
        sh(f"cd {d} && c++ -O3 -Wall -shared -std=c++11 -fPIC "
           f"$({sys.executable} -m pybind11 --includes) mesh_processor.cpp "
           f"-o mesh_processor$({sys.executable}-config --extension-suffix 2>/dev/null || "
           f"python3-config --extension-suffix)", check=False)
        if not glob.glob(f"{d}/mesh_processor*.so"):
            sh(f"cd {d} && {sys.executable} setup.py build_ext --inplace", check=False)
        if cache:
            for f in glob.glob(f"{d}/mesh_processor*.so"):
                shutil.copy(f, cache)
print(f"Hunyuan3D-2 listo ({time.time()-t0:.0f}s)")

# --- Blender: 4.5 LTS (el mas nuevo estable); si no se puede, 4.2 LTS
if not os.path.exists("/content/blender/blender"):
    for ver, serie in (("4.5.3", "4.5"), ("4.5.2", "4.5"), ("4.5.1", "4.5"), ("4.2.3", "4.2")):
        url = f"https://download.blender.org/release/Blender{serie}/blender-{ver}-linux-x64.tar.xz"
        r = subprocess.run(f"wget -q {url} -O /content/blender.tar.xz && tar -xf /content/blender.tar.xz "
                           f"-C /content && mv /content/blender-{ver}-linux-x64 /content/blender",
                           shell=True)
        subprocess.run("rm -f /content/blender.tar.xz", shell=True)
        if r.returncode == 0 and os.path.exists("/content/blender/blender"):
            break

# --- TripoSG (modelo abierto del equipo de Tripo AI): forma de alta fidelidad
if INSTALAR_TRIPOSG and (not os.path.exists("/content/TripoSG") or
                         subprocess.run(f"{sys.executable} -c 'import diso'", shell=True).returncode):
    if not os.path.exists("/content/TripoSG"):
        sh("git clone -q --depth 1 https://github.com/VAST-AI-Research/TripoSG /content/TripoSG", check=False)
    sh(f"{sys.executable} -m pip install -q peft jaxtyping typeguard scikit-image", check=False)
    # diso se compila con CUDA y necesita ver torch: sin aislamiento de build
    r = subprocess.run(f"{sys.executable} -m pip install -q --no-build-isolation diso",
                       shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        r = subprocess.run(f"{sys.executable} -m pip install -q --no-build-isolation "
                           "git+https://github.com/SarahWeiii/diso", shell=True,
                           capture_output=True, text=True)
    print("TripoSG:", "listo" if r.returncode == 0 else
          "diso no compilo (TripoSG desactivado). Error:\n" + (r.stderr or r.stdout)[-800:])
print(sh("/content/blender/blender -b --version | head -1").strip())
print(f"Instalación lista en {time.time()-t0:.0f}s")

## 3. Comprobar GPU, Blender y modelos de IA (carga todo en RAM y GPU)

In [ ]:
# @title 3. Comprobar GPU, descargar modelos, precargarlos en la RAM y cargarlos en la GPU
CARGAR_MULTIVISTA = True  # @param {type:"boolean"}
CARGAR_TEXTURA = True  # @param {type:"boolean"}
MODO_CALIDAD_TAMBIEN = True  # @param {type:"boolean"}
CARGAR_TRIPOSG = True  # @param {type:"boolean"}
CARGAR_TEXTO_A_3D = True  # @param {type:"boolean"}
# @markdown Turbo (5 pasos + FlashVDM) siempre se carga: la forma sale en segundos. Con la casilla de arriba también queda cargado el modelo de 50 pasos para el preajuste "Maxima".
import os, sys, time
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
sys.path.insert(0, "/content/ap3d_app")
sys.path.insert(0, "/content/Hunyuan3D-2")
# Hunyuan3D-2 necesita transformers 4.49: con versiones nuevas su codificador
# de imagen (DINOv2) no carga ("Missing key(s) ... main_image_encoder").
import importlib.metadata as _md, subprocess
if _md.version("transformers") != "4.49.0" or not _md.version("diffusers").startswith("0.32"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.49.0",
                    "diffusers==0.32.2", "accelerate>=1.2,<1.6"], check=True)
    if "transformers" in sys.modules:
        raise SystemExit("Versiones corregidas. Ve a Entorno de ejecución › Reiniciar sesión "
                         "y vuelve a ejecutar la celda 1 y después esta celda 3.")
import torch
from huggingface_hub import snapshot_download
from ap3d import config
from ap3d.engine import get_engine

# Token de Hugging Face (opcional) desde los Secretos de Colab
try:
    from google.colab import userdata
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN") or "")
except Exception:
    pass
if not os.environ.get("HF_TOKEN"):
    os.environ.pop("HF_TOKEN", None)
import logging, warnings
warnings.filterwarnings("ignore")
for _n in ("hy3dgen", "hy3dgen.shapgen", "hy3dgen.shapegen"):
    logging.getLogger(_n).setLevel(logging.WARNING)

# ---- reloj de pasos: duracion de cada uno y fin estimado ----
_T0 = time.time()
_RES = []
_EST = {"Hunyuan3D-2": 120, "Hunyuan3D-2mv": 50, "TripoSG": 45, "Texto a imagen": 90,
        "Precargar pesos en RAM": 15, "Cargar modelos en GPU": 110, "TripoSG en GPU": 25}
_PEND = [k for k in _EST]
def paso(nombre, fn, activo=True):
    if nombre in _PEND:
        _PEND.remove(nombre)
    if not activo:
        _RES.append((nombre, 0, "omitido"))
        return None
    resto = sum(_EST[k] for k in _PEND)
    print(f"[{(time.time()-_T0)/60:4.1f} min] ▶ {nombre}… (despues faltan ~{resto/60:.0f} min)", flush=True)
    t = time.time()
    try:
        r = fn()
        _RES.append((nombre, time.time() - t, "ok"))
        print(f"           ✓ {nombre} en {time.time()-t:.0f} s", flush=True)
        return r
    except Exception as e:
        _RES.append((nombre, time.time() - t, f"fallo: {str(e)[:120]}"))
        print(f"           ⚠ {nombre}: {e}", flush=True)
        return None

print("CUDA:", torch.cuda.is_available())
P = config.detect()
print(config.describe(P))
print(f"Inicio. Primera vez: unos {sum(_EST.values())/60:.0f} min (despues de reiniciar, mucho menos: ya esta descargado).")

pats = ["hunyuan3d-dit-v2-0-turbo/*"] + (["hunyuan3d-dit-v2-0/*"] if MODO_CALIDAD_TAMBIEN else [])
if CARGAR_TEXTURA:
    pats += ["hunyuan3d-paint-v2-0-turbo/*", "hunyuan3d-paint-v2-0/*", "hunyuan3d-delight-v2-0/*"]
paso("Hunyuan3D-2", lambda: snapshot_download("tencent/Hunyuan3D-2", allow_patterns=pats, max_workers=16))
paso("Hunyuan3D-2mv", lambda: snapshot_download("tencent/Hunyuan3D-2mv", allow_patterns=["hunyuan3d-dit-v2-mv-turbo/*"]
     + (["hunyuan3d-dit-v2-mv/*"] if MODO_CALIDAD_TAMBIEN else []), max_workers=16), CARGAR_MULTIVISTA)
def _tripo():
    snapshot_download("VAST-AI/TripoSG", local_dir="/content/TripoSG/pretrained_weights/TripoSG", max_workers=16)
    snapshot_download("briaai/RMBG-1.4", local_dir="/content/TripoSG/pretrained_weights/RMBG-1.4", max_workers=16)
    return True
CARGAR_TRIPOSG = bool(paso("TripoSG", _tripo, CARGAR_TRIPOSG and os.path.exists("/content/TripoSG")))
def _t2i():
    # FLUX pide aceptar su licencia en Hugging Face (con tu token en Secretos de
    # Colab: HF_TOKEN). Sin acceso se usa HunyuanDiT, que no pide nada.
    try:
        snapshot_download("black-forest-labs/FLUX.1-schnell", max_workers=16,
                          ignore_patterns=["flux1-schnell.safetensors", "ae.safetensors", "*.md"])
        return "FLUX"
    except Exception as e:
        print("           FLUX sin acceso (necesita token):", str(e).splitlines()[0][:90])
        print("           -> se usa HunyuanDiT para texto a 3D")
        os.environ["AP3D_T2I"] = "hunyuan"
        snapshot_download("Tencent-Hunyuan/HunyuanDiT-v1.1-Diffusers-Distilled", max_workers=16)
        return "HunyuanDiT"
T2I = paso("Texto a imagen", _t2i, CARGAR_TEXTO_A_3D)

eng = get_engine()
paso("Precargar pesos en RAM", eng.preload_weights, P["preload_weights"])
paso("Cargar modelos en GPU", lambda: eng.load(want_mv=CARGAR_MULTIVISTA, want_paint=CARGAR_TEXTURA,
     want_turbo=True, want_quality=MODO_CALIDAD_TAMBIEN))
paso("TripoSG en GPU", eng.load_triposg, CARGAR_TRIPOSG)

print("\n======== RESUMEN (mandame esto para revisar) ========")
for n, sec, st in _RES:
    print(f"{n:28s} {sec:6.0f} s   {'✅' if st == 'ok' else ('—' if st == 'omitido' else '⚠️')} {st}")
print(f"{'TOTAL':28s} {time.time()-_T0:6.0f} s")
print(f"GPU {P['gpu']} · VRAM usada {torch.cuda.memory_allocated()/2**30:.1f} GB · "
      f"TripoSG {'si' if getattr(eng, 'tripo', None) is not None else 'no'} · "
      f"texto a 3D {T2I or 'no'}")
print("Todo listo. Sigue con la celda 3b (prueba de 1 min) o con la 4/5.")

## 3b. Prueba rápida sin página web (opcional)

In [ ]:
# @title 3b. Prueba corta SIN página web (~1-2 min): confirma que todo funciona y mide tiempos
import glob, sys, time
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
_img = sorted(glob.glob("/content/Hunyuan3D-2/assets/example_images/*.png"))[0]
print("Imagen de prueba:", _img.split("/")[-1])
_t = time.time()
_last = [None]
def _p(f, m):
    if m != _last[0]:
        _last[0] = m
        print(f"[{time.time()-_t:5.0f} s] {int(f*100):3d}%  {m}", flush=True)
_r = get_engine().run(_img, steps=5, octree=384, faces=60000, tex=2048, rig="auto",
                      tex_rapida=True, progress=_p)
_i = _r["info"]
print(f"\n======== RESULTADO (mandame esto para revisar) ========")
print(f"Listo en {time.time()-_t:.0f} s  ->  {_r['zip']}")
print(f"Tiempos: forma {_i.get('segundos_forma')} s · textura {_i.get('segundos_textura')} s · "
      f"Blender {_i.get('segundos_blender')} s")
print(f"Detectado: {(_i.get('clasificacion') or {}).get('tipo', _i.get('rig'))} · huesos {_i.get('huesos')} · "
      f"pesos por {_i.get('pesos')} · malla IA {_i.get('caras_alta')} -> {_i.get('caras')} caras · "
      f"cuadros {_i.get('quads_caras')}")
print("Archivos:", ", ".join(sorted((_i.get('archivos') or {}).values())))
print("Clasificacion (medidas):", _i.get("clasificacion"))
print("UV:", _i.get("uv"))
print("Avisos:", _i.get("avisos") or "ninguno")

## 4. Estudio

In [ ]:
# @title 4. Estudio (dentro del notebook)
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_inline
launch_inline(port=7860)

## 5. Abrir la página web con contraseña

In [ ]:
# @title 5. Abrir la página web con contraseña (para el celular)
CONTRASENA = ""  # @param {type:"string"}
# @markdown Déjala vacía para generar una al azar. El usuario es **ap**. La página deja de funcionar al cerrar la sesión de Colab.
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_web
url, user, pw = launch_web(CONTRASENA, user="ap", port=7861)
print("Página web:", url)
print("Usuario:", user, "· Contraseña:", pw)

## 6. Guardar en Drive

In [ ]:
# @title 6. Guardar todos los resultados en Google Drive
CARPETA = "AP_Model_3D"  # @param {type:"string"}
import os, shutil, glob
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
dst = f"/content/drive/MyDrive/{CARPETA}"
os.makedirs(dst, exist_ok=True)
n = 0
for z in glob.glob("/content/ap3d_salidas/*.zip"):
    if not os.path.exists(os.path.join(dst, os.path.basename(z))):
        shutil.copy(z, dst); n += 1
print(f"{n} modelos nuevos copiados a {dst}")

## 7. Memoria

In [ ]:
# @title 7. (Opcional) Liberar GPU / ver uso de memoria
LIBERAR_MODELOS = False  # @param {type:"boolean"}
import gc, torch, psutil, sys
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
e = get_engine()
if LIBERAR_MODELOS:
    e.shape = e.shape_mv = e.paint = None; e.ready = False
    gc.collect(); torch.cuda.empty_cache()
e.log_mem()